# ⚖️ 21 · Expectation, variance and moments
### Expectation as the long-run average and the centre of gravity · E[h(X)] · the empirical distribution and the sample mean · linearity · variance, standard deviation and the 1/n vs 1/(n−1) question · variance of sums · skewness and kurtosis · conditional expectation and variance, the laws of total expectation and variance · highly variable genes

**BiologyToMath** · Part F — Statistics: from data to estimates · Module 21 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/21_expectation_variance_moments.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib, SymPy (exact integrals and fractions), SciPy (`scipy.stats`, `scipy.integrate` — only to check our own formulas) |
| **Prerequisites** | [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) (random variables, PMF, PDF, CDF, common distributions) and [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb) (joint, marginal and conditional distributions, covariance); [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb) (integrals) |
| **Estimated time** | 170 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Expectation and moments: summarising a distribution by one number; the expected value of a discrete random variable (mean, first moment; two four-sided dice → 5, one six-sided die → 3.5, Bernoulli → p) and of a continuous one (Uniform(−1, 3) → 1); the expectation of a function of a random variable (X ~ Uniform(0, 1), E[X²] = 1/3); the long-run-average and centre-of-gravity interpretations; the empirical distribution and estimating E[X] by the sample mean; properties of expectation (E[aX + b] = aE[X] + b, E[ΣXᵢ] = ΣE[Xᵢ]); summarising gene expression by its average; variance and standard deviation (Var = E[X²] − (E[X])², the fair coin, the buggy coin simulator), estimating variance from data; properties of variance (Var ≥ 0, Var = 0 iff constant, Var(aX + b) = a²Var X, Var of sums of independent variables); variance and differential expression (highly variable genes); conditional expectation and variance (X ~ Uniform(0, 1), Y | X ~ Uniform(X, 1)); covariance and correlation |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All data are **SIMULATED**: dice, coins and the buggy coin simulator; gene expression from the course's infection experiment (`simulate_tissue_expression`) and single-cell counts (`simulate_cell_counts`); the log-normal expression example, the germination counts and the waiting-time density of Section 1 use **illustrative** parameters. The human mutation-rate figure in Section 4 is a rounded, approximate literature value used only for an order-of-magnitude calculation.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Compute** expectations of discrete and continuous random variables by hand and in code (sums, integrals, SymPy),
   and **interpret** them as long-run averages and centres of gravity.
2. **Apply** $E[h(X)] = \sum h(x)p(x)$ or $\int h(x)f(x)\,dx$, and **show** that $E[h(X)] \ne h(E[X])$ in general — the
   reason "mean of the logs" differs from "log of the mean" for expression data.
3. **Estimate** an expectation from data with the empirical distribution and the sample mean.
4. **Prove** linearity of expectation — $E[aX + b] = aE[X] + b$ and $E[\sum X_i] = \sum E[X_i]$ *without* independence —
   and **use** indicator variables to count expected events in DNA.
5. **Derive** $\operatorname{Var}(X) = E[X^2] - (E[X])^2$, $\operatorname{Var}(aX + b) = a^2\operatorname{Var}(X)$ and
   the variance of a sum, and **explain** with simulations why the sample variance divides by $n - 1$.
6. **Compute** skewness and kurtosis and **show** how the log transform tames skewed expression data.
7. **Derive** a conditional expectation and variance step by step, and **verify** the laws of total expectation and
   total variance by calculus and simulation.
8. **Rank** genes by variance to find highly variable genes and **explain** the mean–variance relationship of
   single-cell counts.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, course helpers, the expression and single-cell data | 5 min |
| 1 · Expectation of a discrete random variable | dice, Bernoulli; long-run average; centre of gravity | 20 min |
| 2 · Continuous expectation and $E[h(X)]$ | Uniform(−1, 3); $E[X^2]$; mean of logs vs log of mean | 20 min |
| 3 · Expectation from data | the empirical distribution; average expression of genes | 10 min |
| 4 · Linearity of expectation | proofs; no independence needed; counting CpGs with indicators | 15 min |
| 5 · Variance and standard deviation | the fair coin; the buggy coin simulator; properties | 20 min |
| 6 · Estimating variance: $1/n$ or $1/(n-1)$? | bias, Bessel's correction, `ddof` | 10 min |
| 7 · Variance of sums; covariance revisited | $\operatorname{Var}(aX + b)$, $\operatorname{Var}(X + Y)$, $|\rho| \le 1$ | 15 min |
| 8 · Higher moments | skewness, kurtosis, the log transform | 10 min |
| 9 · Conditional expectation and variance | $Y \mid X \sim U(X, 1)$; laws of total expectation and variance | 25 min |
| 10 · Highly variable genes | variance ranks; mean–variance relationship of counts | 15 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

A single-cell RNA-sequencing experiment measures thousands of genes in thousands of cells. Before any clustering or
differential-expression analysis, two numbers are computed for every gene: its **average expression** and its
**variability** across cells. Genes that are highly expressed are reliable to measure; genes whose expression **varies**
much more than measurement noise alone would produce are the **highly variable genes** — usually the markers that
distinguish cell types — while housekeeping genes, expressed at a constant level in every cell, have almost no variance.
Choosing the highly variable genes is the first step of standard pipelines such as Seurat.

Both numbers are **moments** of a distribution: the **expectation** (the long-run average, the centre of gravity) and
the **variance** (the expected squared distance from the expectation). This module defines them, proves the rules for
manipulating them, shows how to estimate them from data — and why the sample variance divides by $n - 1$ — and ends with
**conditional** expectation and variance, which explain why a population made of several cell types is so much more
variable than any single type.

| Biological question | Mathematical tool | Where |
|---|---|---|
| What is the "typical" number of germinating seeds, or of reads from a gene? | expectation $E[X] = \sum x\,p(x)$ | §1, §3 |
| Is the mean of log-expression the log of the mean expression? | $E[h(X)] \ne h(E[X])$ | §2 |
| How many CG dinucleotides or mutations should we expect in a genome? | linearity of expectation, indicators | §4 |
| How noisy is a measurement? How much do replicates help? | variance, $\operatorname{Var}(\bar X) = \sigma^2/n$ | §5–7 |
| Why log-transform expression data? | skewness | §8 |
| Why are mixtures of cell types so variable? | law of total variance | §9 |
| Which genes mark cell types; which are housekeeping genes? | ranking genes by variance; mean–variance plots | §10 |

> 🧪 **Simulated data, with a known truth.** The expression and single-cell data come from the course simulators, which
> record the true means, the true cell types and which genes are housekeeping genes — so every estimate can be checked.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sympy", "sympy")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fractions import Fraction
from itertools import product

import sympy as sp
from scipy import integrate, stats

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). `simulate_tissue_expression` generates the infection experiment (log2 expression of 400 genes in 24 samples, including the 12 stable `HOUSEKEEPING_GENES` and the infection genes of `INFECTION_PATHWAYS`), and `simulate_cell_counts` generates single-cell RNA counts for four cell types with their `CELL_TYPE_MARKERS` (Poisson, or over-dispersed with `overdispersion > 0`). Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import itertools

import numpy as np
import pandas as pd


# Named genes of the simulated infection experiment.  The infection-response genes are real genes
# known to be induced during infection (cytokines, interferon-stimulated genes, acute-phase
# proteins; lysozyme, LYZ, is an antimicrobial enzyme rather than a classic acute-phase protein and is
# grouped with them only for simplicity); the reference genes are classic "housekeeping" genes used
# to normalise qPCR.
INFECTION_PATHWAYS = {
    "cytokine signalling": ["IL1B", "TNF", "IL6", "CXCL8"],
    "interferon response": ["IRF1", "STAT1", "MX1", "ISG15"],
    "acute phase": ["SAA1", "HAMP", "C3", "LYZ"],
}


HOUSEKEEPING_GENES = ["ACTB", "GAPDH", "EEF1A1", "RPL13A", "TBP", "HPRT1",
                      "PPIA", "UBC", "YWHAZ", "SDHA", "TUBA1A", "B2M"]


def simulate_tissue_expression(n_genes: int = 400, tissues=("spleen", "kidney", "liver"), n_reps: int = 4,
                               seed: int = 2680) -> dict:
    """SIMULATED log2 gene expression from an infection experiment in several tissues.

    Every tissue has ``n_reps`` control and ``n_reps`` infected samples.  The 12 genes of
    ``INFECTION_PATHWAYS`` respond to infection in every tissue (up by 1.5–3 log2 units); further
    "g…" genes respond in one tissue only or in two tissues, so the sets of truly changed genes
    overlap like the differentially-expressed-gene sets of a real multi-tissue study.  The 12
    ``HOUSEKEEPING_GENES`` are stable (tiny noise, no response).  About a third of all genes differ in
    baseline level between tissues.

    Returns a dict with ``X`` (samples × genes, log2 expression), ``samples`` (DataFrame with
    columns sample, tissue, condition), ``genes`` (list), ``true_lfc`` (DataFrame tissues × genes,
    true infected − control difference), ``true_de`` (dict tissue → set of truly changed genes),
    ``pathways`` (dict pathway → genes) and ``source``.
    """
    rng = np.random.default_rng(seed)
    named = [g for genes in INFECTION_PATHWAYS.values() for g in genes]
    n_other = n_genes - len(named) - len(HOUSEKEEPING_GENES)
    if n_other < 60:
        raise ValueError("n_genes must be at least 84")
    genes = named + HOUSEKEEPING_GENES + [f"g{i:03d}" for i in range(1, n_other + 1)]
    J, T = len(genes), len(tissues)
    base = rng.normal(7.0, 1.8, J)
    base[len(named):len(named) + len(HOUSEKEEPING_GENES)] = rng.uniform(9.0, 12.0, len(HOUSEKEEPING_GENES))
    tissue_shift = rng.normal(0.0, 1.0, (T, J)) * (rng.random(J) < 0.35)
    tissue_shift[:, len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.0
    lfc = np.zeros((T, J))
    lfc[:, :len(named)] = rng.uniform(1.5, 3.0, (T, len(named)))
    other = list(range(len(named) + len(HOUSEKEEPING_GENES), J))
    rng.shuffle(other)
    sizes_single = [24, 18, 14, 12, 10][:T]
    pairs = list(itertools.combinations(range(T), 2))
    sizes_pair = [10, 5, 4, 3, 3, 2, 2, 2, 2, 2][:len(pairs)]
    k = 0

    def effect(m):
        return rng.uniform(1.0, 2.5, m) * np.where(rng.random(m) < 0.7, 1.0, -1.0)

    for t, m in enumerate(sizes_single):
        idx = other[k:k + m]
        lfc[t, idx] = effect(m)
        k += m
    for (t1, t2), m in zip(pairs, sizes_pair):
        idx = other[k:k + m]
        e = effect(m)
        lfc[t1, idx] = e
        lfc[t2, idx] = e * rng.uniform(0.8, 1.2, m)
        k += m
    noise_sd = rng.uniform(0.25, 0.55, J)
    noise_sd[len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.08
    rows, X = [], []
    for t, tissue in enumerate(tissues):
        for condition in ("control", "infected"):
            for r in range(1, n_reps + 1):
                mean = base + tissue_shift[t] + (lfc[t] if condition == "infected" else 0.0)
                X.append(mean + rng.normal(0.0, noise_sd))
                rows.append((f"{tissue}_{condition[:3]}{r}", tissue, condition))
    samples = pd.DataFrame(rows, columns=["sample", "tissue", "condition"])
    true_lfc = pd.DataFrame(lfc, index=list(tissues), columns=genes)
    true_de = {tissue: {g for g, v in zip(genes, lfc[t]) if v != 0} for t, tissue in enumerate(tissues)}
    return {"X": np.array(X), "samples": samples, "genes": genes, "true_lfc": true_lfc,
            "true_de": true_de, "pathways": {k_: list(v) for k_, v in INFECTION_PATHWAYS.items()},
            "source": "SIMULATED log2 expression (BiologyToMath simulate_tissue_expression)"}


CELL_TYPE_MARKERS = {
    "T cell": ["CD3E", "CD3D", "IL7R"],
    "B cell": ["CD79A", "MS4A1", "CD19"],
    "macrophage": ["CD68", "LYZ", "CSF1R"],
    "fibroblast": ["COL1A1", "COL1A2", "DCN"],
}


def simulate_cell_counts(n_cells=(160, 120, 90, 70), cell_types=("T cell", "B cell", "macrophage", "fibroblast"),
                         n_background: int = 28, overdispersion: float = 0.0, seed: int = 680) -> dict:
    """SIMULATED single-cell RNA counts (molecules of each gene detected in each cell).

    Each cell type has a true mean count λ for every gene: its three ``CELL_TYPE_MARKERS`` genes are
    high (λ ≈ 6–15) in that type and low (≈ 0.1–0.5) elsewhere; ``n_background`` other genes share
    similar λ (≈ 0.2–4) in all types.  Counts are Poisson(λ); with ``overdispersion`` φ > 0 each
    cell's λ is first multiplied by a Gamma(1/φ, φ) factor (mean 1), which makes counts negative-
    binomial with variance λ + φλ² — closer to real data.

    Returns a dict with ``counts`` (cells × genes, int), ``labels`` (cell type of each cell),
    ``genes``, ``rates`` (DataFrame cell types × genes of true λ) and ``source``.
    """
    rng = np.random.default_rng(seed)
    markers = [g for t in cell_types for g in CELL_TYPE_MARKERS[t]]
    genes = markers + [f"bg{i:02d}" for i in range(1, n_background + 1)]
    rates = np.zeros((len(cell_types), len(genes)))
    bg = rng.uniform(0.2, 4.0, n_background)
    for t, ctype in enumerate(cell_types):
        for j, g in enumerate(markers):
            rates[t, j] = rng.uniform(6.0, 15.0) if g in CELL_TYPE_MARKERS[ctype] else rng.uniform(0.1, 0.5)
        rates[t, len(markers):] = bg * rng.uniform(0.8, 1.25, n_background)
    labels, counts = [], []
    for t, (ctype, n) in enumerate(zip(cell_types, n_cells)):
        lam = np.tile(rates[t], (n, 1))
        if overdispersion > 0:
            lam = lam * rng.gamma(1.0 / overdispersion, overdispersion, size=lam.shape)
        counts.append(rng.poisson(lam))
        labels += [ctype] * n
    return {"counts": np.vstack(counts), "labels": np.array(labels), "genes": genes,
            "rates": pd.DataFrame(rates, index=list(cell_types), columns=genes),
            "source": "SIMULATED single-cell counts (BiologyToMath simulate_cell_counts)"}

print("🧰 toolkit loaded:", ", ".join(sorted(['CELL_TYPE_MARKERS', 'HOUSEKEEPING_GENES', 'INFECTION_PATHWAYS', 'simulate_cell_counts', 'simulate_tissue_expression'])))

In [ ]:
# ── 0.1 The data of this module (all SIMULATED) ──
expr = simulate_tissue_expression()
expr_df = pd.DataFrame(expr["X"], columns=expr["genes"])          # 24 samples × 400 genes, log2 expression
sc = simulate_cell_counts()                                        # Poisson counts
sc_od = simulate_cell_counts(overdispersion=0.3)                  # over-dispersed (negative-binomial-like) counts
counts = pd.DataFrame(sc["counts"], columns=sc["genes"])
print("infection experiment:", expr_df.shape, "| single-cell counts:", counts.shape,
      "| cell types:", pd.Series(sc["labels"]).value_counts().to_dict())
print("first 5 cells × 6 genes (molecules detected):")
print(counts.iloc[:5, [0, 1, 2, 9, 12, 13]])

🔍 **What just happened?** We generated the two data sets of this module. The infection experiment has **24 samples ×
400 genes** (log2 expression). The single-cell data have **440 cells × 40 genes**: 160 T cells, 120 B cells, 90
macrophages and 70 fibroblasts, with three marker genes per cell type (e.g. CD3E for T cells, COL1A1 for fibroblasts) and
28 background genes. Each entry is a **count** of molecules — a discrete random variable, mostly small numbers with many
zeros. Sections 3 and 10 summarise every gene by its mean and its variance.

---
## 1 · The expectation of a discrete random variable

> 📍 **Where we are.** [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) described a random variable by its whole distribution: a PMF, a PDF or a CDF. Often we
> want a **single number** that summarises it. The most important one is the **expectation**.

**A quick recap.** A random variable is a function $X: \Omega \to \mathbb R$; $x$ denotes the values it takes. For a
discrete $X$, $p_X(x) = P(X = x)$ is the **probability mass function** (PMF). $X$ is **continuous** if there is a
function $f_X$ such that $P(a \le X \le b) = \int_a^b f_X(x)\,dx$ for all $a \le b$; $f_X$ is the **probability density
function** (PDF). For any $X$, $F_X(x) = P(X \le x)$ is the **cumulative distribution function**
$F_X: \mathbb R \to [0, 1]$, and $P(X)$ denotes the whole probability distribution of $X$. The next cell redraws the two
standard examples: a Bernoulli PMF with its step-shaped CDF, and a Gaussian PDF with its S-shaped CDF.

In [ ]:
# ── 1.1 Recap: PMF and CDF of a Bernoulli variable; PDF and CDF of a Gaussian ──
p = 0.5
xs = np.linspace(-1, 5, 601)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].step(xs, np.where(xs < 0, 0, np.where(xs < 1, 1 - p, 1)), where="post", color="tab:orange", lw=2,
             label="CDF F(x) = P(X ≤ x)")
axes[0].vlines([0, 1], 0, [1 - p, p], color="gold", lw=3, label="PMF p(x): 1 − p at 0, p at 1")
axes[0].set_xlabel("x"); axes[0].set_ylabel("probability"); axes[0].set_title(f"Bernoulli(p = {p})"); axes[0].legend(loc="lower right")
g = np.linspace(-5, 5, 400)
axes[1].fill_between(g, stats.norm.pdf(g), color="gold", alpha=0.6, label="PDF f(x)")
axes[1].plot(g, stats.norm.cdf(g), color="tab:orange", lw=2, label="CDF F(x)")
axes[1].set_xlabel("x"); axes[1].set_title("standard Gaussian N(0, 1)"); axes[1].legend(loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** For the Bernoulli variable the PMF is two spikes ($1 - p$ at 0 and $p$ at 1) and the CDF is a
staircase that jumps by those amounts. For the Gaussian the PDF is the bell (yellow) and the CDF (orange) is its
accumulated area, rising from 0 to 1. A whole curve is a lot of information; we now compress it into one number.

**Definition.** Given a probability distribution $p(X)$ of a random variable $X$, how can we summarise it with a single
value? The **expectation** is a number that attempts to capture the centre of $p(X)$ and can be interpreted as the
**long-run average of many independent samples** from the distribution. For a discrete random variable $X$, the
**expected value** is

$$E[X] = \sum_{x \in \operatorname{range}(X)} x \cdot p(X = x),$$

*In words:* multiply every possible value by its probability and add up — a **weighted average** of the values, with
the probabilities as weights. Here $\operatorname{range}(X)$ is the set of all possible values that $X$ can take. We
also call $E[X]$ the **mean** or the **first moment** of $X$, and often write $\mu$ or $\mu_X$ for it.

**Example 1.** We roll two independent 4-sided dice (faces 1 to 4) and let $C$ be the sum of the two dice. There are
$4 \times 4 = 16$ equally likely outcomes; we list the possible sums and their probabilities.

In [ ]:
# ── 1.2 Example 1: the sum C of two 4-sided dice — PMF table and E[C] with exact fractions ──
outcomes = list(product(range(1, 5), repeat=2))                      # 16 equally likely (first, second) pairs
rows = []
for c in range(2, 9):
    ways = [f"{a}+{b}" for a, b in outcomes if a + b == c]
    rows.append({"sum C": c, "number of outcomes": f"{len(ways)} ({', '.join(ways)})",
                 "P(C = c)": Fraction(len(ways), 16)})
pmf_C = pd.DataFrame(rows)
print(pmf_C.assign(**{"P(C = c)": pmf_C["P(C = c)"].astype(str)}).to_string(index=False))


def expectation(values, probs):
    """E[X] = Σ x · p(x) for a discrete distribution given as two parallel lists."""
    return sum(x * q for x, q in zip(values, probs))


E_C = expectation(pmf_C["sum C"].tolist(), pmf_C["P(C = c)"].tolist())
terms = " + ".join(f"{c}·{q}" for c, q in zip(pmf_C["sum C"], pmf_C["P(C = c)"].astype(str)))
print(f"\nE[C] = {terms} = {E_C}")
die6 = range(1, 7)
E_D = expectation(die6, [Fraction(1, 6)] * 6)
print(f"one fair 6-sided die: E[D] = (1 + 2 + ... + 6)/6 = {E_D} = {float(E_D)}  — not a possible roll")
for pb in [Fraction(1, 2), Fraction(1, 4)]:
    print(f"Bernoulli(p = {pb}): E[X] = 0·(1 − p) + 1·p = {expectation([0, 1], [1 - pb, pb])}")

🔍 **What just happened?** The PMF of $C$ rises from $1/16$ (only 1+1 gives 2) to $1/4 = 4/16$ (four ways to make 5) and
falls back to $1/16$ (only 4+4 gives 8). Weighting each sum by its probability,

$$E[C] = 2\cdot\tfrac{1}{16} + 3\cdot\tfrac{2}{16} + 4\cdot\tfrac{3}{16} + 5\cdot\tfrac{4}{16} + 6\cdot\tfrac{3}{16} + 7\cdot\tfrac{2}{16} + 8\cdot\tfrac{1}{16} = \tfrac{80}{16} = 5.$$

*In words:* the distribution is symmetric around 5, so its weighted average is 5. In this case the expected value happens
to be one of the outcomes. **Example 2:** for a random variable $D$, the outcome of a single 6-sided die,
$E[D] = 1\cdot\tfrac16 + 2\cdot\tfrac16 + \dots + 6\cdot\tfrac16 = \tfrac{21}{6} = \tfrac72 = 3.5$, **which is not a
possible outcome**: the expected value of a discrete variable need not be an actual outcome. It reflects the average
result over many trials — the sense in which it is used for the long-run behaviour of Markov chains ([Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb)) and for "expected loss" in
decision making. **Bernoulli:** if $X$ takes only the values 0 and 1 with $\Pr(X = 1) = p$, then
$E(X) = 0 \times (1 - p) + 1 \times p = p$ — the expectation of a yes/no variable is the probability of "yes".

### 1.3 The expectation as a long-run average

Roll a fair die again and again and keep track of the running average of the rolls.

In [ ]:
# ── 1.3 Running average of SIMULATED die rolls approaches E[D] = 3.5 ──
rng = np.random.default_rng(SEED + 13)
fig, ax = plt.subplots(figsize=(7, 3.4))
for run, colour in zip(range(3), ["tab:green", "tab:blue", "tab:purple"]):
    rolls = rng.integers(1, 7, size=400)
    running_mean = np.cumsum(rolls) / np.arange(1, 401)
    ax.plot(np.arange(1, 401), running_mean, color=colour, lw=1.2, label=f"run {run + 1}: mean after 400 rolls = {running_mean[-1]:.2f}")
ax.axhline(3.5, color="k", ls="--", lw=1, label="E[D] = 3.5")
ax.set_ylim(1, 6); ax.set_xlabel("number of rolls"); ax.set_ylabel("average of the rolls so far")
ax.set_title("rolling a fair die n times"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each coloured line is one experiment of 400 rolls. Early on the running average jumps around
(after one roll it is simply that roll, anywhere from 1 to 6), but as the number of rolls grows every run settles near
the dashed line at 3.5 — after 400 rolls the three averages are 3.66, 3.31 and 3.36, all within about 0.2 of 3.5. This is the "long-run average"
interpretation of the expectation; [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb) states it precisely as the **law of large numbers** and quantifies how fast
the wobbles shrink.

### 1.4 The expectation as a centre of gravity

The expectation of a random variable — equivalently, the mean of its distribution — can be regarded as the **centre of
gravity** of the distribution. View the $x$-axis as a weightless beam and hang a weight equal to $p(x_j)$ at every
possible value $x_j$. The expected value $E[X]$ is the point at which the beam would balance: the centre of mass of the
probability distribution. For a continuous distribution, the $x$-axis is a rod with mass density $f(x)$, and its centre
of gravity is again $E[X]$. Balance means that the "torques" cancel:

$$\sum_x (x - E[X])\,p(x) = \sum_x x\,p(x) - E[X]\sum_x p(x) = E[X] - E[X]\cdot 1 = 0.$$

*In words:* the probability-weighted distances to the left of $E[X]$ exactly cancel those to the right — which is the
definition of $E[X]$ rearranged.

In [ ]:
# ── 1.4 Balance-beam pictures: a discrete and a continuous distribution (ILLUSTRATIVE parameters) ──
k = np.arange(0, 9)
pmf_seeds = stats.binom.pmf(k, 8, 0.3)                  # seeds germinating out of 8, each with probability 0.3
mean_seeds = np.sum(k * pmf_seeds)
t = np.linspace(0, 14, 400)
dens_wait = stats.gamma.pdf(t, a=2, scale=1.5)            # a right-skewed waiting-time density (hours)
mean_wait = integrate.quad(lambda s: s * stats.gamma.pdf(s, a=2, scale=1.5), 0, np.inf)[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].vlines(k, 0, pmf_seeds, color="tab:blue", lw=3)
axes[0].plot([mean_seeds], [0], marker="^", markersize=14, color="tab:red", clip_on=False)
axes[0].set_xlabel("x = seeds germinating (of 8)"); axes[0].set_ylabel("p(x)")
axes[0].set_title(f"discrete: the beam balances at E[X] = {mean_seeds:.2f}")
axes[1].plot(t, dens_wait, color="tab:blue", lw=2)
axes[1].fill_between(t, dens_wait, color="tab:blue", alpha=0.2)
axes[1].plot([mean_wait], [0], marker="^", markersize=14, color="tab:red", clip_on=False)
axes[1].set_xlabel("x = waiting time (h)"); axes[1].set_ylabel("f(x)")
axes[1].set_title(f"continuous: the rod balances at E[X] = {mean_wait:.2f}")
plt.tight_layout(); plt.show()
print(f"torque check, discrete: Σ (x − E[X]) p(x) = {np.sum((k - mean_seeds) * pmf_seeds):.1e}")
print(f"torque check, continuous: ∫ (x − E[X]) f(x) dx = "
      f"{integrate.quad(lambda s: (s - mean_wait) * stats.gamma.pdf(s, a=2, scale=1.5), 0, np.inf)[0]:.1e}")

🔍 **What just happened?** On the left, the number of seeds that germinate out of 8 (each with probability 0.3) has its
weights piled up at 1, 2 and 3 with a tail to the right; the beam balances at $E[X] = 8 \times 0.3 = 2.40$ (red
triangle). On the right, a skewed waiting-time density balances at $E[X] = 3.00$ hours — to the **right of the peak**,
because the long right tail has leverage. In both cases the weighted distances to the balance point add up to 0 (up to
rounding, $\sim 10^{-16}$). For a skewed distribution the mean is not the most likely value; Section 8 measures that
asymmetry.

✏️ **Your turn.** (1) Compute $E[C]$ for the sum of two **6-sided** dice by hand and with `expectation`. (2) A plant
produces 0, 1, 2 or 3 offspring with probabilities 0.1, 0.3, 0.4, 0.2. What is the expected number of offspring? Is a
population of such plants expected to grow (mean above 1)?

---
## 2 · Continuous expectations and the expectation of a function

> 📍 **Where we are.** The idea of a probability-weighted average generalises to continuous random variables by using
> integrals instead of sums — exactly as probabilities did in [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb).

For a continuous random variable with density $f(x)$, we swap the sum for an integral:

$$E[X] = \int_{-\infty}^{\infty} x \cdot f(x)\,dx.$$

*In words:* weight every value $x$ by its density and add up continuously; the expectation is still called the **mean**
or the **expected value**. The distinction between bounded and unbounded random variables matters here for the same
reason as for sums: if $X$ is bounded the integral is always finite, but for an unbounded $X$ with heavy tails the
integral can fail to converge and the expectation **does not exist** (an example follows).

**Example 3.** $U \sim \text{Uniform}(-1, 3)$. Recall that the density of $\text{Uniform}(a, b)$ is $f(x) = \frac{1}{b - a}$
on $(a, b)$ and 0 elsewhere, so here $f(u) = \tfrac14$ for $u \in (-1, 3)$. Since the density is 0 outside $(-1, 3)$, the
integral runs only from $-1$ to $3$:

$$E[U] = \int_{-1}^{3} x\cdot\frac14\,dx = \frac18 x^2\Big|_{-1}^{3} = \frac98 - \frac18 = 1.$$

*In words:* an antiderivative of $x/4$ is $x^2/8$; evaluating it at the upper limit 3 gives $9/8$ and at the lower limit
$-1$ gives $1/8$; their difference is 1 — the midpoint of the interval, as the centre-of-gravity picture predicts. In
general $E[X] = \frac{a + b}{2}$ for $\text{Uniform}(a, b)$.

In [ ]:
# ── 2.1 E[U] for U ~ Uniform(−1, 3): SymPy, numerical integration and simulation ──
x = sp.symbols("x", real=True)
a_, b_ = sp.symbols("a b", real=True)
print("SymPy: ∫_{-1}^{3} x/4 dx =", sp.integrate(x / 4, (x, -1, 3)))
print("general Uniform(a, b): E =", sp.simplify(sp.integrate(x / (b_ - a_), (x, a_, b_))))
print("scipy.integrate.quad:", round(integrate.quad(lambda u: u * 0.25, -1, 3)[0], 10))
rng = np.random.default_rng(SEED + 21)
print(f"mean of 100,000 SIMULATED draws: {rng.uniform(-1, 3, 100_000).mean():.4f}")

# bounded or not: running means of an exponential (E exists) and a Cauchy variable (E does not exist)
n = 20_000
fig, ax = plt.subplots(figsize=(7, 3.2))
draw_index = np.arange(1, n + 1)
ax.plot(draw_index, np.cumsum(rng.exponential(1.0, n)) / draw_index, label="Exponential(1): E[X] = 1")
ax.plot(draw_index, np.cumsum(rng.standard_cauchy(n)) / draw_index, label="Cauchy: E[X] does not exist")
ax.axhline(1, color="k", ls="--", lw=0.8)
ax.set_xscale("log"); ax.set_ylim(-6, 6); ax.set_xlabel("number of draws (log scale)"); ax.set_ylabel("running mean")
ax.set_title("unbounded variables: a mean may or may not exist (SIMULATED)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** SymPy, numerical integration and 100,000 simulated draws all give $E[U] = 1$ (the simulation to
about two decimals), and SymPy derives the general formula $(a + b)/2$. The figure contrasts two **unbounded** random
variables. The running mean of exponential draws settles at 1, because the exponential's tail is thin enough for
$\int x f(x)\,dx$ to converge. The running mean of **Cauchy** draws (the ratio of two independent standard normals,
whose tails decay only like $1/x^2$) never settles: every so often one enormous draw drags it away, because
$\int |x| f(x)\,dx = \infty$. Biological data are rarely Cauchy, but heavy-tailed measurements (outliers from a faulty
instrument) behave in the same unsettling way — Section 5 meets a mild version.

### 2.2 The expectation of a function of a random variable

Let $Y = h(X)$ be a function of a random variable $X$ — a square, a logarithm, a cost. We do not need the distribution
of $Y$ to find its mean; we can weight $h(x)$ by the distribution of $X$:

$$E(Y) = E\big(h(X)\big) = \sum_x h(x)\cdot p_X(x) \quad\text{(discrete)}, \qquad E\big(h(X)\big) = \int_{-\infty}^{\infty} h(x)\cdot f_X(x)\,dx \quad\text{(continuous)}.$$

*In words:* average the transformed values $h(x)$, each with the probability (or density) of the original value $x$.
(This rule is sometimes called the "law of the unconscious statistician" because it is used without thinking — it can be
proved by grouping the values $x$ with the same $h(x)$.)

**Example.** $Y = X^2$ with $X \sim \text{Uniform}(0, 1)$, whose PDF is $f(x) = 1$ for $0 \le x \le 1$:

$$E[X^2] = \int_0^1 x^2\,dx = \left[\frac{x^3}{3}\right]_0^1 = \frac13.$$

*In words:* the average of the squares of numbers spread uniformly between 0 and 1 is one third. ⚠️ **Common
confusion:** although $E[X] = \tfrac12$, $E[X^2]$ is **not** $(E[X])^2 = \tfrac14$. In general $E[h(X)] \ne h(E[X])$;
equality holds for linear functions $h(x) = ax + b$ (Section 4). For $h(x) = x^2$ the gap $E[X^2] - (E[X])^2 =
\tfrac13 - \tfrac14 = \tfrac1{12}$ is never negative — it is the **variance** of Section 5.

In [ ]:
# ── 2.2 E[X²] for X ~ Uniform(0, 1), and the "mean of logs vs log of mean" trap for expression data ──
print("E[X²] = ∫_0^1 x² dx =", sp.integrate(x ** 2, (x, 0, 1)), "| (E[X])² =", sp.integrate(x, (x, 0, 1)) ** 2)
rng = np.random.default_rng(SEED + 22)
u = rng.random(100_000)
print(f"SIMULATED: mean of X² = {np.mean(u ** 2):.4f},  (mean of X)² = {np.mean(u) ** 2:.4f}")

# ILLUSTRATIVE: log2 expression of a gene in 100,000 cells ~ N(5, 1); raw expression = 2^(log2 expression)
log2_expr = rng.normal(5.0, 1.0, 100_000)
raw_expr = 2.0 ** log2_expr
print(f"\nmean of log2 expression E[log2 X] = {log2_expr.mean():.3f}")
print(f"log2 of mean raw expression log2 E[X] = {np.log2(raw_expr.mean()):.3f}"
      f"   (theory for log-normal data: 5 + σ²·ln2/2 = {5 + np.log(2) / 2:.3f})")

🔍 **What just happened?** SymPy gives $E[X^2] = 1/3$ while $(E[X])^2 = 1/4$, and the simulated averages (0.333 versus
0.249) agree. The second example matters every day in genomics: if log2 expression is normal with mean 5, then the
**mean of the logs** is 5.0, but the **log of the mean raw expression** is about 5.35 — averaging before or after the
log transform gives different "average expression levels". Because $2^y$ is convex (curves upward), a few very high
values pull the raw mean up (Jensen's inequality). Always say which one you computed.

✏️ **Your turn.** (1) For a fair 6-sided die compute $E[D^2]$ by hand ($\sum d^2/6$) and compare it with $(E[D])^2$.
(2) For $U \sim \text{Uniform}(-1, 3)$ compute $E[U^2]$ with SymPy. Is it larger or smaller than $(E[U])^2 = 1$?

### 📘 Lecture notes — expectation

**Definition.** Discrete: $E[X] = \sum_{x \in \operatorname{range}(X)} x\cdot p(X = x)$ ($\operatorname{range}(X)$ = all
possible values); continuous: $E[X] = \int_{-\infty}^{\infty} x\cdot f(x)\,dx$. Also called the **mean**, the
**expected value** or the **first moment**. Interpretations: the long-run average of many independent samples; the
centre of gravity of the distribution (the balance point of a beam/rod loaded with $p(x)$ or $f(x)$).

**Worked examples.** Two 4-sided dice: $E[C] = 2\cdot\tfrac1{16} + 3\cdot\tfrac2{16} + 4\cdot\tfrac3{16} + 5\cdot\tfrac4{16}
+ 6\cdot\tfrac3{16} + 7\cdot\tfrac2{16} + 8\cdot\tfrac1{16} = \tfrac{80}{16} = 5$. One 6-sided die: $E[D] = \tfrac{21}{6} =
3.5$ — not a possible outcome. Bernoulli($p$): $E(X) = 0\times(1-p) + 1\times p = p$. $U \sim \text{Uniform}(-1, 3)$,
$f = \tfrac14$ on $(-1, 3)$: $E[U] = \int_{-1}^{3} \tfrac{x}{4}\,dx = \tfrac18x^2\big|_{-1}^{3} = \tfrac98 - \tfrac18 = 1$.

**Function of a random variable.** $E[h(X)] = \sum_x h(x)p_X(x)$ or $\int h(x)f_X(x)\,dx$. For $X \sim \text{Uniform}(0, 1)$:
$E[X^2] = \int_0^1 x^2\,dx = \tfrac13$, whereas $(E[X])^2 = \tfrac14$.

**Common confusions.** $E[h(X)] \ne h(E[X])$ unless $h$ is linear. Integrate only over the region where the density is
positive. For heavy-tailed unbounded variables the expectation may not exist.

> 🎓 **Exam tip.** Write the PMF table (values and probabilities) before computing $E[X]$, and check that the
> probabilities add up to 1 — most wrong expectations come from a wrong PMF, not from the arithmetic.

---
## 3 · Expectation from data: the empirical distribution and the sample mean

> 📍 **Where we are.** In biology the distribution $P(X)$ is almost never known. What we have are **data**:
> independent, identically distributed (**i.i.d.**) draws $X_1, \dots, X_n$ from $P(X)$ — $n$ samples. How do we get an
> expectation out of them?

**The empirical distribution.** Normalising the histogram of the data by $n$ gives the **empirical distribution**
$\hat P(X)$: each observed value gets probability (number of times it occurs)/$n$. **Example:** let $X$ be the outcome of
a single coin toss, coded $X = 1$ for H and $X = 0$ for T, and suppose we observe $n = 8$ samples: H, H, T, T, H, T, T, T.
The empirical distribution is $\hat P(X = 1) = 3/8$, $\hat P(X = 0) = 5/8$, and its expectation is

$$E_{\hat P}[X] = 1\cdot\tfrac38 + 0\cdot\tfrac58 = \tfrac38 = \frac{1}{n}\sum_{i=1}^{n} X_i.$$

*In words:* the expectation of the empirical distribution is exactly the **sample mean** of the data. When $P(X)$ is
unknown we therefore approximate

$$E[X] \approx \bar X = \frac{1}{n}\sum_{i=1}^{n} X_i,$$

*In words:* the average of i.i.d. draws estimates the expectation of their distribution. This sample mean forms the basis
for statistical inference ([Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb)–[Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb)); the approximation improves as $n$ grows (Section 1.3).

In [ ]:
# ── 3.1 Empirical distribution of 8 coin tosses; average expression of genes across SIMULATED cells ──
tosses = ["H", "H", "T", "T", "H", "T", "T", "T"]
x_coded = np.array([1 if s == "H" else 0 for s in tosses])
print("empirical distribution:", {v: str(Fraction(int(np.sum(x_coded == v)), len(x_coded))) for v in (1, 0)},
      "| expectation under it = sample mean =", Fraction(int(x_coded.sum()), len(x_coded)))

# single-cell counts: the sample mean of each gene across the 440 cells estimates E[X_gene]
labels = pd.Series(sc["labels"])
share = labels.value_counts(normalize=True)                         # proportion of each cell type
true_mean = (sc["rates"].mul(share, axis=0)).sum(axis=0)            # E[X] = Σ_type P(type) · λ(type, gene)
summary = pd.DataFrame({"sample mean": counts.mean(), "true E[X]": true_mean}).round(3)
print("\nmost highly expressed genes (sample mean of counts across 440 cells):")
print(summary.sort_values("sample mean", ascending=False).head(6))
print(f"largest |sample mean − true E[X]| over all 40 genes: {np.abs(summary['sample mean'] - summary['true E[X]']).max():.3f}")

🔍 **What just happened?** The eight tosses give $\hat P(\text{H}) = 3/8$, and the expectation of this empirical
distribution is the sample mean $3/8 = 0.375$ — our best guess for the coin's probability of heads from so few tosses.
For the single-cell data, the **average expression** of a gene is the sample mean $\frac1n\sum_i x_i$ of its counts
across cells, where $x_i$ is the count in cell $i$. The most highly expressed genes are a mixture of background genes,
expressed at similar levels in every cell (bg13, bg14, …), and markers of the larger cell populations (CD19 of the 120
B cells, CD3D of the 160 T cells); every sample mean is within 0.14 of the true expectation stored by the simulator. ⚠️ The true expectation of a randomly chosen cell's count is a **mixture**,
$E[X] = \sum_{\text{type}} P(\text{type})\,\lambda_{\text{type}}$ — Section 9 proves this (the law of total expectation).

### 🌉 Bridge — average expression and normalisation in RNA-seq

Averages are the first summary of any RNA-seq data set: they are used to **identify highly expressed genes** and to
**normalise** expression levels for downstream analysis (Anders *et al.*, 2013). A subtle point: a cell with twice as
many sequenced molecules has (roughly) twice as many counts for every gene, so raw averages mix biology with sequencing
depth. Normalisation divides each cell's counts by a size factor before averaging — using linearity, $E[X/s] = E[X]/s$
(Section 4). Single-cell pipelines ([BiologyToML 30 · Single-cell RNA-seq](../../../../ml/tutorial/BiologyToML/notebooks/30_single_cell_rnaseq.ipynb)) and count models ([BiologyToML 24 · GLMs for count data](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb)) build on exactly these expectations.

---
## 4 · Properties of expectation: linearity

> 📍 **Where we are.** Expectations would be of limited use if we had to recompute an integral for every new
> quantity. Two rules — both forms of **linearity** — let us compute expectations of sums and rescaled variables from
> the expectations of their parts.

**Linear function.** If $Y = aX + b$, where $a$ and $b$ are finite constants, then $E(Y) = aE(X) + b$.

*Proof.* Assume first, for convenience, that $X$ has a continuous distribution with PDF $f$. Then

$$E(Y) = E(aX + b) = \int_{-\infty}^{\infty}(ax + b)f(x)\,dx = a\int_{-\infty}^{\infty}x f(x)\,dx + b\int_{-\infty}^{\infty}f(x)\,dx = aE(X) + b.$$

*In words:* split the integral of a sum into a sum of integrals and pull the constants out; the first integral is
$E(X)$ by definition and the second is 1 because $f$ is a density. A similar proof works for a discrete distribution
(replace $\int \dots dx$ by $\sum_x$). **Example.** Suppose $E(X) = 5$. Then $E(3X - 5) = 3E(X) - 5 = 10$ and
$E(-3X + 15) = -3E(X) + 15 = 0$.

**Sums.** If $X_1, \dots, X_n$ are $n$ random variables such that each expectation $E(X_i)$ is finite, then

$$E(X_1 + \dots + X_n) = E(X_1) + \dots + E(X_n).$$

*In words:* the expectation of a sum is the sum of the expectations. *Proof* for $n = 2$, assuming for convenience that
$X_1$ and $X_2$ have a continuous joint distribution with joint PDF $f$ ([Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb)):

$$\begin{aligned}
E(X_1 + X_2) &= \int_{-\infty}^{\infty}\!\!\int_{-\infty}^{\infty}(x_1 + x_2)\,f(x_1, x_2)\,dx_1\,dx_2 \\
&= \int\!\!\int x_1 f(x_1, x_2)\,dx_1\,dx_2 + \int\!\!\int x_2 f(x_1, x_2)\,dx_1\,dx_2 \\
&= \int_{-\infty}^{\infty} x_1\Big(\int_{-\infty}^{\infty} f(x_1, x_2)\,dx_2\Big)dx_1 + \int_{-\infty}^{\infty} x_2\Big(\int_{-\infty}^{\infty} f(x_1, x_2)\,dx_1\Big)dx_2 \\
&= \int_{-\infty}^{\infty} x_1 f_1(x_1)\,dx_1 + \int_{-\infty}^{\infty} x_2 f_2(x_2)\,dx_2 = E(X_1) + E(X_2),
\end{aligned}$$

*In words:* expand $(x_1 + x_2)$ into two double integrals; in each, integrate first over the variable that is *not*
multiplied in front, which turns the joint density into a **marginal** density $f_1$ or $f_2$; what is left are the two
expectations. The proof for a discrete distribution is similar (double sums, and marginals by summing), and the case of
$n$ variables follows by adding one variable at a time (induction).

⚠️ **Common confusion — no independence needed.** Nowhere did the proof assume that $f(x_1, x_2) = f_1(x_1)f_2(x_2)$.
Linearity of expectation holds for **any** random variables, dependent or not. (Some summaries state it "for
independent random variables" together with the variance rule of Section 7 — independence is needed only for the
variance.)

In [ ]:
# ── 4.1 Linearity checks: E[aX + b], and E[X + Y] for strongly DEPENDENT variables (SIMULATED) ──
rng = np.random.default_rng(SEED + 41)
X = rng.integers(1, 7, 1_000_000)                  # die rolls, E[X] = 3.5
print(f"E[3X − 5]: simulated {np.mean(3 * X - 5):.3f}  vs  3·3.5 − 5 = {3 * 3.5 - 5}")
Y_opposite = 7 - X                                  # the face on the bottom of the die: completely dependent on X
Y_square = X ** 2                                   # also completely dependent on X
print(f"E[X + (7 − X)] = {np.mean(X + Y_opposite):.3f}  vs  E[X] + E[7 − X] = {np.mean(X):.3f} + {np.mean(Y_opposite):.3f}")
print(f"E[X + X²]      = {np.mean(X + Y_square):.3f}  vs  E[X] + E[X²]   = {np.mean(X):.3f} + {np.mean(Y_square):.3f}"
      f"   (exact 3.5 + 91/6 = {3.5 + 91 / 6:.3f})")
x1, x2 = sp.symbols("x1 x2", real=True)
f_joint = x1 + x2                                   # the dependent joint density on the unit square from Module 19
lhs = sp.integrate((x1 + x2) * f_joint, (x1, 0, 1), (x2, 0, 1))
rhs = sp.integrate(x1 * sp.integrate(f_joint, (x2, 0, 1)), (x1, 0, 1)) + sp.integrate(x2 * sp.integrate(f_joint, (x1, 0, 1)), (x2, 0, 1))
print(f"joint density x1 + x2 (dependent): E[X1 + X2] = {lhs},  E[X1] + E[X2] via marginals = {rhs}")

🔍 **What just happened?** With a million simulated die rolls, $E[3X - 5] \approx 5.5 = 3 \cdot 3.5 - 5$. The bottom face
$7 - X$ is completely determined by the top face, yet $E[X + (7 - X)] = 7 = 3.5 + 3.5$; and $E[X + X^2] = 3.5 + 91/6
\approx 18.667$ although $X^2$ is a function of $X$. For the dependent joint density $f(x_1, x_2) = x_1 + x_2$ of
[Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb), SymPy computes $E[X_1 + X_2] = 7/6$ directly from the double integral and gets the same $7/12 + 7/12$ from the
marginals — exactly the steps of the proof.

### 4.2 Counting with indicators: expected CpGs and expected mutations

Linearity makes counting easy. Write a count as a **sum of indicator variables** $\mathbb 1_k$ (1 if event $k$ happens,
0 otherwise); since $E[\mathbb 1_k] = 1\cdot P(\text{event } k) + 0 = P(\text{event } k)$ (a Bernoulli mean),

$$E\Big[\sum_k \mathbb 1_k\Big] = \sum_k P(\text{event } k).$$

*In words:* the expected number of events is the sum of their probabilities — whether or not the events are independent.
**Example:** in a DNA sequence of length $L$ with independent, uniform letters, how many times do we expect to see the
dinucleotide CG? There are $L - 1$ neighbouring pairs, each equal to CG with probability $\tfrac14\cdot\tfrac14 =
\tfrac1{16}$, so $E[\#\text{CG}] = (L - 1)/16$ — even though overlapping pairs are **not** independent (a pair CG makes
the next pair start with G). The same reasoning gives the expected number of new mutations in a genome: $L$ sites times
the per-site mutation rate.

In [ ]:
# ── 4.2 Expected number of CG dinucleotides in SIMULATED uniform DNA; expected new mutations per genome ──
rng = np.random.default_rng(SEED + 42)
L, n_seqs = 1000, 2000
seqs = rng.integers(0, 4, size=(n_seqs, L))                       # letters coded A=0, C=1, G=2, T=3
n_CG = np.sum((seqs[:, :-1] == 1) & (seqs[:, 1:] == 2), axis=1)   # CG count in each sequence
print(f"average number of CG in {n_seqs} sequences of length {L}: {n_CG.mean():.2f}  vs  (L − 1)/16 = {(L - 1) / 16:.2f}")
both_CGs = np.mean(((seqs[:, :-2] == 1) & (seqs[:, 1:-1] == 2)) & ((seqs[:, 1:-1] == 1) & (seqs[:, 2:] == 2)))
print(f"overlapping pairs are dependent: P(pair k = CG and pair k+1 = CG) = {both_CGs:.4f}  vs  (1/16)² = {1 / 256:.4f}")
genome_size, mutation_rate = 3.1e9, 1.2e-8        # approximate haploid human genome; ~1.2e-8 per site per generation
print(f"expected new mutations per haploid human genome per generation ≈ {genome_size * mutation_rate:.0f} "
      f"(≈ {2 * genome_size * mutation_rate:.0f} per diploid child)")

🔍 **What just happened?** The 2,000 simulated sequences contain on average 62.65 CG dinucleotides, close to
$(1000 - 1)/16 = 62.44$ (the difference is sampling noise). The second line shows that the indicators are dependent: two overlapping pairs can never both be
CG (that would need a letter to be both G and C), so the joint probability is 0 instead of $1/256$ — and linearity did
not care. For mutations, $3.1 \times 10^9$ sites times a rate of about $1.2 \times 10^{-8}$ per site per generation gives
about 37 new mutations per haploid genome, about 74 per child — the right order of magnitude for the de-novo mutation
counts measured by sequencing parent–child trios. (Both rounded inputs are approximate.)

✏️ **Your turn.** (1) If $E(X) = 2$ and $E(Y) = -1$, compute $E(4X - 2Y + 7)$. (2) In background DNA modelled as a Markov
chain ([Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb)), $P(\text{C followed by G}) = P(C)\cdot 0.08$. Use linearity to write the expected number of CG pairs in a
sequence of length $L$ in terms of the base composition. (3) A binomial variable is the sum of $n$ Bernoulli($\alpha$)
indicators. Use linearity to show $E[X] = n\alpha$ ([Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb)).

### 📘 Lecture notes — properties of expectation

**Linear function.** $E(aX + b) = aE(X) + b$ for constants $a, b$. Proof (continuous): $\int(ax + b)f(x)\,dx =
a\int xf(x)\,dx + b\int f(x)\,dx = aE(X) + b$; similar for discrete. Example: $E(X) = 5 \Rightarrow E(3X - 5) = 10$,
$E(-3X + 15) = 0$.

**Sums.** If each $E(X_i)$ is finite, $E(X_1 + \dots + X_n) = E(X_1) + \dots + E(X_n)$. Proof ($n = 2$, joint density $f$):
$\iint(x_1 + x_2)f\,dx_1dx_2 = \int x_1 f_1(x_1)dx_1 + \int x_2 f_2(x_2)dx_2$ with the marginals $f_1, f_2$.
**No independence is required.**

**Indicators.** $E[\#\text{events}] = \sum_k P(\text{event}_k)$; e.g. $E[\#\text{CG}] = (L - 1)/16$ in uniform random DNA;
binomial mean $n\alpha$.

**Data.** Empirical distribution = normalised histogram; its expectation is the sample mean, $E[X] \approx \frac1n\sum X_i$
(H, H, T, T, H, T, T, T → $\hat P(H) = 3/8$, sample mean $3/8$). The average expression of a gene across cells,
$\frac1n\sum_i x_i$, is such an estimate.

> 🎓 **Exam tip.** If a question says "assume independence" for an expectation of a sum, the assumption is not needed —
> but it *is* needed (or a zero covariance) for the variance of the sum. Say so explicitly.

---
## 5 · Variance and standard deviation

> 📍 **Where we are.** Two distributions can have the same mean and look completely different. In addition to the mean
> we often want to know how **far** the likely outcomes are from it — for a normal distribution, how narrow the bell is.

The **variance** of a random variable is

$$\operatorname{Var}[X] = E\big[(X - E[X])^2\big].$$

*In words:* the expected **squared** difference between the random variable and its mean. Squaring makes every
deviation positive (so left and right deviations do not cancel, as they did in the balance equation of Section 1.4) and
weights large deviations heavily.

**Alternative method for calculating the variance.** For every random variable $X$, $\operatorname{Var}(X) = E(X^2) -
[E(X)]^2$. *Proof.* Let $E(X) = \mu$. Then

$$\operatorname{Var}(X) = E[(X - \mu)^2] = E(X^2 - 2\mu X + \mu^2) = E(X^2) - 2\mu E(X) + \mu^2 = E(X^2) - 2\mu^2 + \mu^2 = E(X^2) - \mu^2.$$

*In words:* expand the square, use linearity of expectation term by term ($\mu$ is a constant), and use $E(X) = \mu$.
"The mean of the square minus the square of the mean" — often the quickest way to compute a variance, and the reason
$E[X^2] \ge (E[X])^2$ always.

**Standard deviation.** The standard deviation is the square root of the variance,

$$\sigma(X) = \sqrt{\operatorname{Var}[X]} = \sqrt{E\big[(X - E[X])^2\big]},$$

*In words:* undo the squaring, so that the spread is measured **in the same units as $X$** (mg/dL, counts, log2 units)
rather than in squared units.

**Example — a fair coin flip.** Let $F = 1$ for heads and $F = 0$ for tails, so $E[F] = \tfrac12$. The difference
between $F$ and $E[F]$ is $+\tfrac12$ or $-\tfrac12$, with equal probability, and thus

$$\operatorname{Var}[F] = E\big[(F - E[F])^2\big] = \tfrac12\left(+\tfrac12\right)^2 + \tfrac12\left(-\tfrac12\right)^2 = \tfrac14, \qquad \sigma[F] = \sqrt{\operatorname{Var}[F]} = \sqrt{\tfrac14} = \tfrac12.$$

*In words:* the coin is always exactly half a unit away from its mean, so its standard deviation is one half.

In [ ]:
# ── 5.1 Variances by the definition and by the shortcut: coin, die, Bernoulli, Uniform(0, 1) ──
def variance(values, probs):
    """Var[X] = Σ (x − E[X])² p(x) for a discrete distribution (exact with Fractions)."""
    mu = expectation(values, probs)
    return sum((v - mu) ** 2 * q for v, q in zip(values, probs))


half, sixth = Fraction(1, 2), Fraction(1, 6)
print("fair coin:  Var by definition =", variance([0, 1], [half, half]),
      "| shortcut E[F²] − E[F]² =", expectation([0, 1], [half, half]) - expectation([0, 1], [half, half]) ** 2)
var_D = variance(range(1, 7), [sixth] * 6)
print(f"fair die:   Var = {var_D} = {float(var_D):.4f}, σ = {np.sqrt(float(var_D)):.4f}"
      f" | shortcut 91/6 − (7/2)² = {Fraction(91, 6) - Fraction(7, 2) ** 2}")
alpha = Fraction(3, 10)
print("Bernoulli(0.3): Var =", variance([0, 1], [1 - alpha, alpha]), "= α(1 − α) =", alpha * (1 - alpha))
print("Uniform(0, 1): Var = E[X²] − E[X]² =", sp.integrate(x ** 2, (x, 0, 1)) - sp.integrate(x, (x, 0, 1)) ** 2)

🔍 **What just happened?** Both formulas agree in every case: the fair coin has variance $1/4$; a fair die has variance
$35/12 \approx 2.92$ and standard deviation about 1.71 (via the shortcut, $91/6 - 49/4 = 35/12$); a Bernoulli($\alpha$)
variable has variance $\alpha(1 - \alpha)$, here $21/100$ — largest for $\alpha = 1/2$, when the outcome is most
uncertain; and Uniform(0, 1) has variance $1/12$, the "gap" of Section 2.2.

### 5.2 A simulation with a mistake

What if we tried to simulate a coin flip into a variable $S$, but made a mistake: now we get 1 or 0 with probability
$0.4995$ each, but in some cases we output **100** (with probability $0.001$)? What are $E[S]$, $\operatorname{Var}[S]$
and $\sigma[S]$?

$$E[S] = 0\cdot 0.4995 + 1\cdot 0.4995 + 100\cdot 0.001 = 0 + 0.4995 + 0.1 = 0.5995,$$

*In words:* the rare 100 shifts the mean only slightly, from 0.5 to about 0.6.

$$\operatorname{Var}[S] = E\big[(S - E[S])^2\big] = 0.4995\cdot 0.5995^2 + 0.4995\cdot 0.4005^2 + 0.001\cdot 99.4005^2 \approx 0.180 + 0.080 + 9.880 \approx 10.14,$$

*In words:* the three possible deviations from the mean are $-0.5995$ (for 0), $+0.4005$ (for 1) and $+99.4005$ (for
100); squared and weighted, the single rare value contributes **9.880 of the 10.14** — almost all of the variance. Thus
$\sigma[S] = \sqrt{\operatorname{Var}[S]} \approx \sqrt{10.14} \approx 3.184$, more than six times the standard
deviation of a correct coin.

In [ ]:
# ── 5.2 The buggy coin simulator S: exact moments, and what samples of different sizes reveal (SIMULATED) ──
s_values = [0, 1, 100]
s_probs = [Fraction(4995, 10000), Fraction(4995, 10000), Fraction(1, 1000)]
E_S, Var_S = expectation(s_values, s_probs), variance(s_values, s_probs)
contrib = [q * (v - E_S) ** 2 for v, q in zip(s_values, s_probs)]
print(f"E[S] = {float(E_S)},  Var[S] = {float(Var_S):.6f}  (contributions {[round(float(c), 3) for c in contrib]}),"
      f"  σ[S] = {np.sqrt(float(Var_S)):.4f}")
print(f"shortcut: E[S²] − E[S]² = {float(expectation([v ** 2 for v in s_values], s_probs))} − {float(E_S) ** 2:.6f}"
      f" = {float(expectation([v ** 2 for v in s_values], s_probs) - E_S ** 2):.6f}")
for n in [100, 1000, 100_000]:
    print(f"P(no 100 at all in {n:>7,} draws) = 0.999^{n} = {0.999 ** n:.4f}")

rng = np.random.default_rng(SEED + 52)
p_float = [float(q) for q in s_probs]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, n, reps in [(axes[0], 100, 2000), (axes[1], 100_000, 200)]:
    sample_vars = np.array([rng.choice(s_values, size=n, p=p_float).var() for _ in range(reps)])
    ax.hist(sample_vars, bins=40, color="lightgray", edgecolor="gray")
    ax.axvline(float(Var_S), color="tab:red", lw=2, label="true Var[S] = 10.14")
    ax.axvline(0.25, color="tab:blue", lw=2, ls="--", label="Var of a correct coin = 0.25")
    ax.set_xlabel("sample variance"); ax.set_ylabel("number of experiments")
    ax.set_title(f"{reps:,} experiments of n = {n:,} draws:\n{np.mean(sample_vars < 1):.0%} look like a correct coin")
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The exact calculation confirms $E[S] = 0.5995$, $\operatorname{Var}[S] = 10.1401$ (the
contributions 0.180, 0.080 and 9.880) and $\sigma[S] = 3.1844$; the shortcut $E[S^2] - E[S]^2 = 10.4995 - 0.3594$ gives
the same value. So clearly the variance is much higher — but how often do we actually expect to **see** a value of 100?
With $p(S = 100) = 0.001$, that is about one in every 1,000 draws; if we draw much fewer than that, we may think it is a
fair coin. With $n = 100$ draws there is a 90.5 % chance of never seeing a 100, and in the left panel most of the
2,000 simulated experiments have a sample variance near 0.25 — they "look like" a correct coin — while the rest jump to
very large values when a single 100 appears. With $n = 100{,}000$ draws (right panel) every experiment sees about a hundred
100s and the sample variance clusters around the true 10.14. This leads to the question of how we **measure** these
things in practice — how large a sample is needed to estimate a mean or a variance reliably — which is the subject of
[Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb). Biologically: one faulty well or one contaminated sample can dominate a variance.

### 5.3 Two properties of the variance

**(1) $\operatorname{Var}(X) \ge 0$ for every $X$; if $X$ is bounded, $\operatorname{Var}(X)$ exists and is finite.**
*Proof.* $\operatorname{Var}(X)$ is the mean of the non-negative random variable $(X - \mu)^2$, and the mean of a
non-negative variable cannot be negative. If $X$ is bounded, its mean exists, hence the variance is defined; and if $X$
is bounded then so is $(X - \mu)^2$, so the variance is finite.

**(2) $\operatorname{Var}(X) = 0$ if and only if there is a constant $c$ with $\Pr(X = c) = 1$.**
*Proof.* ($\Leftarrow$) If $\Pr(X = c) = 1$, then $E(X) = c$ and $\Pr[(X - c)^2 = 0] = 1$, so $\operatorname{Var}(X) =
E[(X - c)^2] = 0$. ($\Rightarrow$) Suppose $\operatorname{Var}(X) = 0$ and let $\mu = E(X)$. For a discrete $X$,
$0 = \sum_x (x - \mu)^2\,p(x)$ is a sum of **non-negative** terms, so every term must be 0: for every $x \ne \mu$ we must
have $p(x) = 0$. Hence all the probability sits at $\mu$: $\Pr(X = \mu) = 1$. (The same idea works in general: a
non-negative random variable with mean 0 is 0 with probability 1, so $\Pr[(X - \mu)^2 = 0] = 1$.)

*In words:* a variance is never negative, and it is zero only for a quantity that never varies — the mathematical
definition of a "constant" measurement such as an ideal housekeeping gene (Section 10).

✏️ **Your turn.** (1) Change the buggy simulator so that the 100 appears with probability 0.0001 (and 0 and 1 with
probability 0.49995 each). Compute $E[S]$ and $\operatorname{Var}[S]$ exactly. (2) How many draws do you need to have a
95 % chance of seeing at least one 100? (*Hint:* solve $1 - 0.9999^n = 0.95$.)

### 📘 Lecture notes — variance and standard deviation

**Definitions.** $\operatorname{Var}[X] = E\big[(X - E[X])^2\big]$ — the expected squared difference from the mean;
$\sigma(X) = \sqrt{\operatorname{Var}[X]}$ — spread in the units of $X$. Shortcut:
$\operatorname{Var}(X) = E(X^2) - [E(X)]^2$ (proof: expand $(X - \mu)^2$ and use linearity).

**Examples.** Fair coin: $E[F] = \tfrac12$, deviations $\pm\tfrac12$, $\operatorname{Var}[F] = \tfrac12(\tfrac12)^2 +
\tfrac12(-\tfrac12)^2 = \tfrac14$, $\sigma[F] = \tfrac12$. Fair die: $35/12$. Bernoulli: $\alpha(1 - \alpha)$.
Uniform(0, 1): $1/12$. **Buggy simulator** ($S = 0$ or 1 with probability 0.4995 each, 100 with probability 0.001):
$E[S] = 0.5995$; $\operatorname{Var}[S] = 0.4995\cdot0.5995^2 + 0.4995\cdot0.4005^2 + 0.001\cdot99.4005^2 \approx 0.180 +
0.080 + 9.880 \approx 10.14$; $\sigma[S] \approx 3.184$. A value seen once in 1,000 draws dominates the variance, yet in
100 draws it is missing 90 % of the time.

**Properties.** $\operatorname{Var}(X) \ge 0$, finite if $X$ is bounded; $\operatorname{Var}(X) = 0 \iff \Pr(X = c) = 1$
for some constant $c$.

**Estimating from data.** $\operatorname{Var}(X) \approx \frac1n\sum_{i}(X_i - \bar X)^2$ with $\bar X = \frac1n\sum X_i$ —
but see Section 6 for why statisticians divide by $n - 1$.

> 🎓 **Exam tip.** When a variance comes out negative, you have made an arithmetic error — usually forgetting to square
> the mean in $E[X^2] - (E[X])^2$. Report the standard deviation with the units of the data.

---
## 6 · Estimating variance from data: divide by $n$ or by $n - 1$?

> 📍 **Where we are.** Just as the sample mean estimates $E[X]$, a sample variance estimates $\operatorname{Var}(X)$
> when the true distribution is unknown — which is how it is used widely in real-world statistics. There is a subtlety.

The natural estimate replaces both expectations by averages:

$$\hat\sigma^2_{n} = \frac{1}{n}\sum_{i=1}^{n}(X_i - \bar X)^2, \qquad \bar X = \frac{1}{n}\sum_{i=1}^{n}X_i.$$

*In words:* the average squared distance of the data points from **their own** mean. Because $\bar X$ is computed from
the same data, it sits closer to the data than the true mean $\mu$ does, so the squared distances are, on average, a bit
too small. Precisely, using the identity $\sum_i (X_i - \bar X)^2 = \sum_i (X_i - \mu)^2 - n(\bar X - \mu)^2$ (expand
$X_i - \mu = (X_i - \bar X) + (\bar X - \mu)$ and use $\sum_i (X_i - \bar X) = 0$):

$$E\Big[\sum_{i}(X_i - \bar X)^2\Big] = \sum_i E\big[(X_i - \mu)^2\big] - n\,E\big[(\bar X - \mu)^2\big] = n\sigma^2 - n\cdot\frac{\sigma^2}{n} = (n - 1)\sigma^2.$$

*In words:* each of the $n$ terms $(X_i - \mu)^2$ has expectation $\sigma^2$, and $E[(\bar X - \mu)^2] = \operatorname{Var}
(\bar X) = \sigma^2/n$ for independent draws (proved in Section 7). So dividing by $n$ gives on average
$\frac{n - 1}{n}\sigma^2$ — too small — whereas dividing by $n - 1$ gives exactly $\sigma^2$ on average. The **sample
variance** with **Bessel's correction**,

$$s^2 = \frac{1}{n - 1}\sum_{i=1}^{n}(X_i - \bar X)^2,$$

*In words:* the same sum of squared deviations divided by one less than the number of data points; it is
**unbiased** ($E[s^2] = \sigma^2$). One way to remember it: the $n$ deviations $X_i - \bar X$ add up to 0, so only $n - 1$
of them are free ("$n - 1$ degrees of freedom").

In [ ]:
# ── 6.1 The bias of the 1/n estimator, by simulation (SIMULATED normal data with σ² = 4) ──
rng = np.random.default_rng(SEED + 61)
sigma2 = 4.0
rows = []
for n in [2, 3, 5, 10, 30]:
    data = rng.normal(0.0, np.sqrt(sigma2), size=(200_000, n))      # 200,000 data sets of size n
    v_n = data.var(axis=1, ddof=0)                                  # divide by n
    v_n1 = data.var(axis=1, ddof=1)                                 # divide by n − 1
    rows.append([n, v_n.mean(), (n - 1) / n * sigma2, v_n1.mean()])
print(pd.DataFrame(rows, columns=["n", "average of 1/n estimate", "theory (n−1)/n·σ²", "average of 1/(n−1) estimate"]).round(3).to_string(index=False))

sample = np.array([7.1, 6.4, 7.9, 6.8])                             # four log2 expression measurements
print(f"\nfour replicates {sample}: np.var (ddof=0) = {np.var(sample):.4f}, np.var(ddof=1) = {np.var(sample, ddof=1):.4f},"
      f" pandas .var() = {pd.Series(sample).var():.4f}")

🔍 **What just happened?** For each sample size we generated 200,000 data sets from a distribution with true variance 4.
Dividing by $n$ underestimates on average by exactly the predicted factor $(n - 1)/n$: about 2.0 for $n = 2$ (half the
truth!), about 3.2 for $n = 5$, about 3.87 for $n = 30$. Dividing by $n - 1$ is right on average for every $n$. The
correction matters most for the small sample sizes typical of biology (three or four replicates per condition). ⚠️
**Software trap:** NumPy's `np.var` and `np.std` divide by $n$ by default (`ddof=0`), while pandas' `.var()` and `.std()`
divide by $n - 1$ (`ddof=1`) — the same four replicates give 0.3025 or 0.4033 depending on the library.

### 🌉 Bridge — the sample variance in tests and in machine learning

$s^2$ with $n - 1$ is the variance used in every t-test and confidence interval ([Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb), [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb)): the "degrees of
freedom" of the t-distribution are exactly that $n - 1$. In 02-620, the 1/n version reappears as the **maximum-likelihood
estimate** of a Gaussian variance — biased, but the MLE ([Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb), [BiologyToML 05 · Likelihood, MLE and MAP](../../../../ml/tutorial/BiologyToML/notebooks/05_likelihood_mle_map.ipynb)) — and the difference between the two is the
simplest example of the bias–variance trade-off. In linear regression the unbiased noise variance divides by $n - p$,
one degree of freedom lost per fitted coefficient ([Module 27 · 📉 Linear regression: the whole course in one model](27_linear_regression_review.ipynb)).

---
## 7 · Variance of linear functions and sums; covariance revisited

> 📍 **Where we are.** Linearity told us how expectations combine. Variances combine differently: constants come out
> **squared**, shifts disappear, and sums pick up **covariance** terms.

**Linearity.** For a random variable $X$ and constants $a, b \in \mathbb R$:

$$E[aX + b] = aE[X] + b, \qquad \operatorname{Var}[aX + b] = a^2\operatorname{Var}[X].$$

*Proof of the second.* $E[aX + b] = a\mu + b$, so $\operatorname{Var}[aX + b] = E\big[(aX + b - a\mu - b)^2\big] =
E\big[a^2(X - \mu)^2\big] = a^2E[(X - \mu)^2] = a^2\operatorname{Var}[X]$. *In words:* shifting by $b$ moves the whole
distribution without changing its spread; scaling by $a$ multiplies every deviation by $a$ and every squared deviation by
$a^2$ (so the standard deviation by $|a|$). Converting log2 expression to natural logs multiplies by $\ln 2$, so the
variance is multiplied by $(\ln 2)^2 \approx 0.48$.

**Summation.** For two random variables, expanding the square,

$$\operatorname{Var}[X + Y] = E\big[((X - \mu_X) + (Y - \mu_Y))^2\big] = \operatorname{Var}[X] + \operatorname{Var}[Y] + 2\operatorname{Cov}(X, Y),$$

*In words:* the variance of a sum is the sum of the variances **plus twice the covariance** — the cross term
$2E[(X - \mu_X)(Y - \mu_Y)]$ of the expanded square. In general
$\operatorname{Var}\big[\sum_i X_i\big] = \sum_i \operatorname{Var}[X_i] + 2\sum_{i<j}\operatorname{Cov}(X_i, X_j)$. For a set
$X_1, X_2, \dots, X_n$ of **independent** random variables all covariances are 0 ([Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb)), so

$$\operatorname{Var}\Big[\sum_{i \in [n]} X_i\Big] = \sum_{i \in [n]}\operatorname{Var}[X_i],$$

*In words:* for independent variables — and only then, or more generally for uncorrelated ones — variances add.
Combining both rules, the **mean of $n$ independent replicates** with variance $\sigma^2$ each has
$\operatorname{Var}(\bar X) = \operatorname{Var}\big(\tfrac1n\sum X_i\big) = \tfrac1{n^2}\cdot n\sigma^2 = \tfrac{\sigma^2}{n}$:
averaging four replicates halves the standard deviation.

In [ ]:
# ── 7.1 Variances of sums: independent, perfectly dependent, and real (SIMULATED) genes ──
rng = np.random.default_rng(SEED + 71)
D1, D2 = rng.integers(1, 7, 1_000_000), rng.integers(1, 7, 1_000_000)   # two independent dice
print(f"Var(D1) ≈ {D1.var():.3f} (exact 35/12 = {35 / 12:.3f})")
print(f"independent: Var(D1 + D2) ≈ {(D1 + D2).var():.3f}  vs  Var(D1) + Var(D2) = {2 * 35 / 12:.3f}")
print(f"same die:    Var(D1 + D1) = Var(2·D1) ≈ {(2 * D1).var():.3f}  vs  4·Var(D1) = {4 * 35 / 12:.3f}")
print(f"opposite:    Var(D1 + (7 − D1)) = {(D1 + 7 - D1).var():.3f}  (the covariance −35/12 cancels everything)")
print(f"Var(3·D1 + 10) ≈ {(3 * D1 + 10).var():.3f}  vs  9·Var(D1) = {9 * 35 / 12:.3f}")

stat1, mx1 = expr_df["STAT1"], expr_df["MX1"]
lhs = (stat1 + mx1).var()
rhs = stat1.var() + mx1.var() + 2 * stat1.cov(mx1)
print(f"\nSTAT1 + MX1 over 24 samples: Var(sum) = {lhs:.3f}; Var + Var = {stat1.var() + mx1.var():.3f}; "
      f"+ 2·Cov = {rhs:.3f}")
reps = rng.normal(7.0, 0.5, size=(100_000, 4))                       # 4 technical replicates, σ = 0.5
print(f"mean of 4 replicates: SD ≈ {reps.mean(axis=1).std():.3f}  vs  σ/√4 = {0.5 / 2:.3f}")

🔍 **What just happened?** For two **independent** dice the variances add: $\operatorname{Var}(D_1 + D_2) \approx 5.83 =
2 \times 35/12$. For the **same** die counted twice the sum is $2D_1$, with variance $4 \times 35/12 \approx 11.67$ — the
covariance $\operatorname{Cov}(D_1, D_1) = \operatorname{Var}(D_1)$ adds $2 \times 2.92$. For the top and bottom faces the
covariance is $-35/12$ and the sum is always 7, so its variance is exactly 0. Scaling by 3 multiplies the variance by 9;
adding 10 changes nothing. In the infection data, STAT1 and MX1 are strongly positively correlated, so the variance of
their sum (4.955) is much larger than the sum of their variances (2.825); adding $2\operatorname{Cov}$ recovers it exactly
— the identity holds for sample quantities too. Finally, averaging 4 independent replicates with $\sigma = 0.5$ gives a
standard deviation of 0.25: the $\sigma/\sqrt n$ law that [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb) builds on.

### 7.2 Covariance and correlation revisited

When we are interested in the joint distribution of two random variables, it is useful to have a summary of how much
they **depend on each other**. Covariance and correlation are attempts to measure that dependence, but they only capture
a particular type of dependence, namely **linear** dependence. Given two random variables $X$ and $Y$ with expectations
$\mu_X = E[X]$ and $\mu_Y = E[Y]$, the **covariance** is

$$\operatorname{Cov}(X, Y) = E[(X - \mu_X)(Y - \mu_Y)] = E[XY - \mu_Y X - \mu_X Y + \mu_X\mu_Y] = E[XY] - \mu_Y E[X] - \mu_X E[Y] + \mu_X\mu_Y = E[XY] - E[X]E[Y],$$

*In words:* expand, apply linearity, and use $E[X] = \mu_X$, $E[Y] = \mu_Y$: the two middle terms each equal
$E[X]E[Y]$ and one of them cancels the last term. ⚠️ Keep track of which mean multiplies which variable: the term
$\mu_Y X$ gives $\mu_Y E[X]$ and $\mu_X Y$ gives $\mu_X E[Y]$. With standard deviations $\sigma_X$ and $\sigma_Y$, the
**correlation** — a normalised version of the covariance — is

$$\rho(X, Y) = \frac{\operatorname{Cov}(X, Y)}{\sigma_X\sigma_Y}.$$

*In words:* the covariance measured in units of the two standard deviations. **Why $-1 \le \rho \le 1$:** apply
"variance is never negative" to the standardised variables: $0 \le \operatorname{Var}\big(\tfrac{X}{\sigma_X} \pm
\tfrac{Y}{\sigma_Y}\big) = 1 + 1 \pm 2\rho$, so $\rho \ge -1$ and $\rho \le 1$; equality forces the variance to be 0,
i.e. $Y$ is an exact linear function of $X$. And because only *linear* co-variation is measured, $\rho$ can be 0 for
strongly dependent variables ($Y = X^2$, [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb)).

In [ ]:
# ── 7.2 Covariance two ways, correlation, and the bound |ρ| ≤ 1 on SIMULATED data ──
rng = np.random.default_rng(SEED + 72)
Xc = rng.normal(0, 1, 100_000)
for name, Yc in [("Y = 2X + noise", 2 * Xc + rng.normal(0, 1, 100_000)),
                 ("Y = −X (exactly linear)", -Xc),
                 ("Y = X² (non-linear)", Xc ** 2)]:
    cov_def = np.mean((Xc - Xc.mean()) * (Yc - Yc.mean()))
    cov_short = np.mean(Xc * Yc) - Xc.mean() * Yc.mean()
    rho = cov_def / (Xc.std() * Yc.std())
    v_plus = np.var(Xc / Xc.std() + Yc / Yc.std())
    v_minus = np.var(Xc / Xc.std() - Yc / Yc.std())
    print(f"{name:25s}: Cov = {cov_def:7.3f} (shortcut {cov_short:7.3f}), ρ = {rho:6.3f}, "
          f"Var(X/σX + Y/σY) = {v_plus:.3f}, Var(X/σX − Y/σY) = {v_minus:.3f}")

🔍 **What just happened?** The definition and the shortcut $E[XY] - E[X]E[Y]$ give identical covariances. For
$Y = 2X + \text{noise}$, $\rho \approx 0.89$ (a strong but imperfect linear relation); for $Y = -X$, $\rho = -1$ exactly
and $\operatorname{Var}(X/\sigma_X + Y/\sigma_Y) = 2 + 2\rho = 0$ — the bound is attained only by an exact linear relation;
for $Y = X^2$ the correlation is about 0 although $Y$ is completely determined by $X$. In every row the two standardised
variances are $2 \pm 2\rho$ and never negative, which is the whole proof of $|\rho| \le 1$.

✏️ **Your turn.** (1) Prove $\operatorname{Var}(X - Y) = \operatorname{Var}(X) + \operatorname{Var}(Y) -
2\operatorname{Cov}(X, Y)$. Why does this make **paired** designs (measuring the same patients before and after a
treatment) powerful when the two measurements are positively correlated? (2) qPCR normalises a target gene by a
reference gene: $\Delta C_t = C_t^{\text{target}} - C_t^{\text{ref}}$. If both have variance 0.04 and correlation 0.8,
what is $\operatorname{Var}(\Delta C_t)$?

### 📘 Lecture notes — properties of variance; covariance and correlation

**Linearity.** $E[aX + b] = aE[X] + b$; $\operatorname{Var}[aX + b] = a^2\operatorname{Var}[X]$ (shifts do not change spread;
scales enter squared).

**Summation.** For **independent** $X_1, \dots, X_n$: $E\big[\sum X_i\big] = \sum E[X_i]$ (true even without
independence) and $\operatorname{Var}\big[\sum X_i\big] = \sum\operatorname{Var}[X_i]$ (needs independence or zero
covariances). In general $\operatorname{Var}[X + Y] = \operatorname{Var}[X] + \operatorname{Var}[Y] +
2\operatorname{Cov}(X, Y)$. Mean of $n$ independent replicates: $\operatorname{Var}(\bar X) = \sigma^2/n$.

**Covariance and correlation.** $\operatorname{Cov}(X, Y) = E[(X - \mu_X)(Y - \mu_Y)] = E[XY] - E[X]E[Y]$ (derivation:
$E[XY - \mu_YX - \mu_XY + \mu_X\mu_Y] = E[XY] - \mu_YE[X] - \mu_XE[Y] + \mu_X\mu_Y$). $\rho(X, Y) =
\operatorname{Cov}(X, Y)/(\sigma_X\sigma_Y) \in [-1, 1]$, a normalised covariance; it measures only **linear** dependence.

**Sample variance.** $\frac1n\sum(X_i - \bar X)^2$ has expectation $\frac{n-1}{n}\sigma^2$; $s^2 = \frac1{n-1}\sum(X_i -
\bar X)^2$ is unbiased (Bessel's correction).

> 🎓 **Exam tip.** "Var(X + Y) = Var X + Var Y" is the most common wrong step in exams: write the covariance term first,
> then argue why it is zero.

---
## 8 · Higher moments: skewness and kurtosis

> 📍 **Where we are.** The mean is the first moment and the variance is the second **central** moment. Higher moments
> describe the **shape** of a distribution — its asymmetry and the weight of its tails — which matters for choosing
> transformations and tests.

The **$k$-th moment** of $X$ is $E[X^k]$ and the **$k$-th central moment** is $E[(X - \mu)^k]$: the mean is the first
moment, the variance the second central moment. Dividing the third and fourth central moments by the matching power of
$\sigma$ gives two unit-free shape numbers:

$$\text{skewness } \gamma_1 = \frac{E[(X - \mu)^3]}{\sigma^3}, \qquad \text{kurtosis } \kappa = \frac{E[(X - \mu)^4]}{\sigma^4}.$$

*In words:* **skewness** measures asymmetry — cubing keeps the sign of each deviation, so a long right tail gives
$\gamma_1 > 0$ and a symmetric distribution gives $\gamma_1 = 0$; **kurtosis** measures how heavy the tails are —
fourth powers make rare large deviations dominate. A normal distribution has $\gamma_1 = 0$ and $\kappa = 3$; the
**excess kurtosis** $\kappa - 3$ compares tails with the normal (SciPy reports excess kurtosis by default). Reference
values: Uniform $\gamma_1 = 0, \kappa = 1.8$ (light tails); Exponential $\gamma_1 = 2, \kappa = 9$ (long right tail).

Gene-expression measurements on the raw scale are typically strongly **right-skewed**: most genes are expressed at low
levels and a few at very high levels. The log transform is used precisely to remove this skew.

In [ ]:
# ── 8.1 Skewness and kurtosis from the definitions, checked with SciPy; raw vs log2 expression (SIMULATED) ──
def shape_moments(data):
    """(skewness, kurtosis) of a sample: third and fourth central moments divided by σ³ and σ⁴."""
    d = np.asarray(data, float) - np.mean(data)
    sd = np.sqrt(np.mean(d ** 2))
    return np.mean(d ** 3) / sd ** 3, np.mean(d ** 4) / sd ** 4


rng = np.random.default_rng(SEED + 81)
for name, draws in [("normal", rng.normal(size=500_000)), ("uniform", rng.random(500_000)),
                    ("exponential", rng.exponential(size=500_000))]:
    sk, ku = shape_moments(draws)
    print(f"{name:12s}: skewness {sk:6.3f}, kurtosis {ku:6.3f} | scipy: {stats.skew(draws):6.3f}, "
          f"{stats.kurtosis(draws, fisher=False):6.3f}")

log2_values = expr_df.to_numpy().ravel()            # all 24 × 400 log2 expression values
raw_values = 2.0 ** log2_values                     # back on the raw (linear) scale
sk_raw, ku_raw = shape_moments(raw_values)
sk_log, ku_log = shape_moments(log2_values)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(raw_values, bins=80, color="lightgray", edgecolor="gray")
axes[0].set_xlabel("raw expression 2^(log2 value)"); axes[0].set_ylabel("number of gene × sample values")
axes[0].set_title(f"raw scale: skewness {sk_raw:.1f}, kurtosis {ku_raw:.0f}")
axes[1].hist(log2_values, bins=80, color="lightgray", edgecolor="gray")
axes[1].set_xlabel("log2 expression"); axes[1].set_title(f"log2 scale: skewness {sk_log:.2f}, kurtosis {ku_log:.2f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Our two-line definitions reproduce SciPy's values and the reference numbers: the normal sample
has skewness ≈ 0 and kurtosis ≈ 3, the uniform ≈ 0 and ≈ 1.8, the exponential ≈ 2 and ≈ 9. On the raw scale the 9,600
expression values of the infection experiment are extremely right-skewed (skewness 12.2, kurtosis 233): a long tail
of highly expressed genes (the largest value is about 30,000, the median only about $2^{7} = 128$) dominates means and variances. After the
log2 transform the distribution is nearly symmetric (skewness 0.02) with normal-like kurtosis (3.00). That is why expression is analysed on a
log scale — and why "mean expression" must say which scale it refers to (Section 2.2).

✏️ **Your turn.** (1) Compute the skewness of a Bernoulli(0.1) variable from the definition and check the formula
$(1 - 2p)/\sqrt{p(1 - p)}$. (2) Compute the skewness of the single-cell counts of the background gene `bg01` and of
$\log(1 + \text{count})$. Does the log help for counts too?

---
## 9 · Conditional expectation and conditional variance

> 📍 **Where we are.** A conditional distribution $p(x \mid y)$ ([Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb)) is a distribution, so it has a mean and a
> variance. They answer questions such as "what is the expected expression of a gene **in T cells**?" and, through two
> beautiful laws, connect group-wise summaries with the summary of the whole population.

**Discrete.** The conditional expectation and variance of $X$ given $Y = y$ are

$$E[X \mid Y = y] = \sum_x x\cdot p(X = x \mid Y = y), \qquad \operatorname{Var}[X \mid Y = y] = \sum_x \big(x - E[X \mid Y = y]\big)^2\,p(X = x \mid Y = y).$$

*In words:* exactly the usual mean and variance, computed with the conditional distribution — "the mean and the spread of
$X$ among the cases where $Y = y$".

**Continuous.**

$$E[X \mid Y = y] = \int_{-\infty}^{\infty} x\cdot f(x \mid y)\,dx, \qquad \operatorname{Var}[X \mid Y = y] = \int_{-\infty}^{\infty}\big(x - E[X \mid Y = y]\big)^2 f(x \mid y)\,dx.$$

*In words:* the same with the conditional density $f(x \mid y) = f(x, y)/f_Y(y)$. **A discrete example** from [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb):
toss a fair coin three times, $X$ = heads in the first toss, $Y$ = total heads. Given $X = 1$, $Y = 1 + (\text{heads in
two more tosses})$ with conditional distribution $(0, \tfrac14, \tfrac12, \tfrac14)$ on $y = 0, 1, 2, 3$, so
$E[Y \mid X = 1] = 1\cdot\tfrac14 + 2\cdot\tfrac12 + 3\cdot\tfrac14 = 2$ and, similarly, $E[Y \mid X = 0] = 1$; both
conditional variances are $\tfrac12$.

### 9.1 A continuous worked example

Assume $X \sim \text{Uniform}(0, 1)$ and, given $X = x$, $Y \mid X = x \sim \text{Uniform}(x, 1)$: first pick a random
point $x$ in $[0, 1]$, then a random point $Y$ between $x$ and 1. The conditional density is
$f(y \mid x) = \frac{1}{1 - x}$ for $x < y < 1$. **Conditional expectation**, every line:

$$\begin{aligned}
E[Y \mid X = x] &= \int_{-\infty}^{\infty} y\cdot f(y \mid x)\,dy = \int_x^1 y\cdot\frac{1}{1 - x}\,dy && \text{(the density is 0 outside } (x, 1))\\
&= \frac{1}{1 - x}\int_x^1 y\,dy = \frac{1}{1 - x}\left[\frac12 y^2\right]_x^1 && \Big(\textstyle\int_x^1 y\,dy = \big[\tfrac12y^2\big]_x^1\Big)\\
&= \frac{1}{1 - x}\left[\frac12 - \frac12x^2\right] = \frac{1 - x^2}{2(1 - x)} = \frac{1 + x}{2} && \big(1 - x^2 = (1 - x)(1 + x)\big)
\end{aligned}$$

*In words:* the conditional mean is the midpoint of the interval $(x, 1)$, as it must be for a uniform distribution.

**Conditional variance**, every line (write $m = \frac{1 + x}{2}$ for the conditional mean):

$$\begin{aligned}
\operatorname{Var}[Y \mid X = x] &= \int_{-\infty}^{\infty}\big(y - E[Y \mid X = x]\big)^2 f(y \mid x)\,dy = \int_x^1\left[y - \frac{1 + x}{2}\right]^2\cdot\frac{1}{1 - x}\,dy \\
&= \frac{1}{1 - x}\int_x^1\frac{\big(2y - (1 + x)\big)^2}{4}\,dy = \frac{1}{1 - x}\int_x^1\frac{4y^2 - 4(1 + x)y + (1 + x)^2}{4}\,dy \\
&= \frac{1}{1 - x}\int_x^1\left(y^2 - (1 + x)y + \frac{(1 + x)^2}{4}\right)dy \\
&= \frac{1}{1 - x}\left[\frac{y^3}{3} - \frac{1 + x}{2}y^2 + \frac{(1 + x)^2}{4}y\right]_{y = x}^{y = 1}.
\end{aligned}$$

*In words:* write the squared deviation over a common denominator, expand the square, divide by 4, and integrate term by
term ($\int y^2 = \tfrac13y^3$, $\int(1 + x)y = \tfrac{1 + x}{2}y^2$, $\int\tfrac{(1 + x)^2}{4} = \tfrac{(1 + x)^2}{4}y$ —
remember $x$ is a constant inside the $dy$ integral). Now evaluate at $y = 1$ and subtract the value at $y = x$:

$$\begin{aligned}
\operatorname{Var}[Y \mid X = x] &= \frac{1}{1 - x}\left(\left[\frac{1}{3} - \frac{1 + x}{2} + \frac{(1 + x)^2}{4}\right] - \left[\frac{x^3}{3} - \frac{1 + x}{2}x^2 + \frac{(1 + x)^2}{4}x\right]\right) \\
&= \frac{1}{1 - x}\left(\frac{1 - x^3}{3} - \frac{1 + x}{2}(1 - x^2) + \frac{(1 + x)^2}{4}(1 - x)\right) && \text{(group by powers)}\\
&= \frac{1}{1 - x}\left(\frac{(1 - x)(1 + x + x^2)}{3} - \frac{(1 + x)^2(1 - x)}{2} + \frac{(1 + x)^2(1 - x)}{4}\right) && \text{(factor out } 1 - x)\\
&= \frac{1 + x + x^2}{3} - \frac{(1 + x)^2}{2} + \frac{(1 + x)^2}{4} = \frac{1 + x + x^2}{3} - \frac{(1 + x)^2}{4} \\
&= \frac{4(1 + x + x^2) - 3(1 + 2x + x^2)}{12} = \frac{1 - 2x + x^2}{12} = \frac{(1 - x)^2}{12}.
\end{aligned}$$

*In words:* the factorisations $1 - x^3 = (1 - x)(1 + x + x^2)$ and $1 - x^2 = (1 - x)(1 + x)$ let us cancel the
$\frac{1}{1 - x}$; putting the two remaining fractions over 12 leaves $(1 - x)^2/12$ — the variance of a uniform
distribution on an interval of length $1 - x$ (recall $\operatorname{Var}\,\text{Uniform}(a, b) = (b - a)^2/12$). **A
shortcut** gives the same in two lines: $E[Y^2 \mid X = x] = \frac{1}{1 - x}\int_x^1 y^2\,dy = \frac{1 - x^3}{3(1 - x)} =
\frac{1 + x + x^2}{3}$, and $\operatorname{Var}[Y \mid X = x] = E[Y^2 \mid X = x] - (E[Y \mid X = x])^2 = \frac{1 + x +
x^2}{3} - \frac{(1 + x)^2}{4}$.

In [ ]:
# ── 9.1 Verify E[Y | X = x] and Var[Y | X = x] with SymPy and with SIMULATED pairs ──
xs_, ys_ = sp.symbols("x y", real=True)
f_cond = 1 / (1 - xs_)                                                 # density of Y given X = x on (x, 1)
E_cond = sp.simplify(sp.integrate(ys_ * f_cond, (ys_, xs_, 1)))
V_cond = sp.factor(sp.simplify(sp.integrate((ys_ - E_cond) ** 2 * f_cond, (ys_, xs_, 1))))
E2_cond = sp.simplify(sp.integrate(ys_ ** 2 * f_cond, (ys_, xs_, 1)))
print("E[Y | X = x]   =", E_cond)
print("Var[Y | X = x] =", V_cond, "| shortcut E[Y²|x] − E[Y|x]² =", sp.factor(sp.simplify(E2_cond - E_cond ** 2)),
      "| E[Y²|x] =", E2_cond)

rng = np.random.default_rng(SEED + 91)
X_sim = rng.random(200_000)
Y_sim = rng.uniform(X_sim, 1.0)                                         # Y | X = x ~ Uniform(x, 1)
bins = np.linspace(0, 1, 21)
which = np.digitize(X_sim, bins) - 1
mid = (bins[:-1] + bins[1:]) / 2
cond_mean = np.array([Y_sim[which == b].mean() for b in range(20)])
cond_var = np.array([Y_sim[which == b].var() for b in range(20)])
print(f"largest |binned mean − (1 + x)/2| = {np.abs(cond_mean - (1 + mid) / 2).max():.4f}; "
      f"largest |binned var − (1 − x)²/12| = {np.abs(cond_var - (1 - mid) ** 2 / 12).max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].scatter(X_sim[:1500], Y_sim[:1500], s=3, color="0.6", label="SIMULATED (X, Y)")
gx = np.linspace(0, 1, 100)
axes[0].plot(gx, (1 + gx) / 2, color="tab:red", lw=2, label="E[Y | X = x] = (1 + x)/2")
axes[0].plot(mid, cond_mean, "o", color="tab:blue", ms=4, label="mean of Y in each x-bin")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y"); axes[0].set_title("conditional expectation"); axes[0].legend(fontsize=7)
axes[1].plot(gx, (1 - gx) ** 2 / 12, color="tab:red", lw=2, label="Var[Y | X = x] = (1 − x)²/12")
axes[1].plot(mid, cond_var, "o", color="tab:blue", ms=4, label="variance of Y in each x-bin")
axes[1].set_xlabel("x"); axes[1].set_ylabel("conditional variance"); axes[1].set_title("conditional variance")
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** SymPy reproduces both results: $E[Y \mid X = x] = x/2 + 1/2$ and $\operatorname{Var}[Y \mid X = x]
= (x - 1)^2/12$ (the same as $(1 - x)^2/12$), and the shortcut via $E[Y^2 \mid x] = (x^2 + x + 1)/3$ agrees. In the
simulation we grouped 200,000 pairs into 20 bins of $x$ and computed the mean and variance of $Y$ within each bin: the
blue dots sit on the red curves (largest deviations of a few thousandths — the bins have width 0.05, so $x$ is not quite
constant within a bin). The conditional mean rises with $x$ while the conditional variance shrinks to 0 as the interval
$(x, 1)$ collapses.

### 9.2 The laws of total expectation and total variance

Since $E[Y \mid X = x]$ is a function of $x$, plugging in the random $X$ gives a **random variable** $E[Y \mid X]$. Two
laws relate it to the unconditional moments:

$$E[Y] = E\big[E[Y \mid X]\big], \qquad \operatorname{Var}(Y) = E\big[\operatorname{Var}(Y \mid X)\big] + \operatorname{Var}\big(E[Y \mid X]\big).$$

*In words:* **the overall mean is the average of the group means** (law of total expectation); **the overall variance is
the average within-group variance plus the variance between the group means** (law of total variance). *Proof of the
first* (discrete): $E[E[Y \mid X]] = \sum_x E[Y \mid X = x]\,p(x) = \sum_x\sum_y y\,p(y \mid x)\,p(x) = \sum_y y\sum_x p(x,
y) = \sum_y y\,p(y) = E[Y]$ — product rule, swap the sums, marginalise. The second follows by applying the first to $Y^2$
and using $\operatorname{Var} = E[\cdot^2] - E[\cdot]^2$ twice.

For our example: $E[Y] = E\big[\tfrac{1 + X}{2}\big] = \tfrac{1 + 1/2}{2} = \tfrac34$ (linearity, $E[X] = \tfrac12$), and

$$\operatorname{Var}(Y) = E\Big[\frac{(1 - X)^2}{12}\Big] + \operatorname{Var}\Big(\frac{1 + X}{2}\Big) = \frac{1}{12}\cdot\frac13 + \frac14\cdot\frac{1}{12} = \frac{1}{36} + \frac{1}{48} = \frac{7}{144} \approx 0.0486.$$

*In words:* $E[(1 - X)^2] = E[X'^2] = \tfrac13$ because $1 - X$ is again Uniform(0, 1); $\operatorname{Var}(\tfrac{1 + X}{2}) =
\tfrac14\operatorname{Var}(X) = \tfrac14\cdot\tfrac1{12}$ by the scaling rule of Section 7. We can check both laws against
the marginal density of $Y$: integrating $x$ out of $f(x, y) = 1\cdot\frac{1}{1 - x}$ on $0 < x < y$ gives
$f_Y(y) = \int_0^y \frac{dx}{1 - x} = -\ln(1 - y)$.

In [ ]:
# ── 9.2 Laws of total expectation and variance: SymPy and simulation ──
print("d/dx(−ln(1 − x)) =", sp.diff(-sp.log(1 - xs_), xs_), " → f_Y(y) = ∫_0^y dx/(1 − x) = −ln(1 − y)")
u_ = sp.symbols("u", positive=True)                             # substitute u = 1 − y (keeps SymPy on the real logarithm)
f_U = -sp.log(u_)                                               # f_Y written in terms of u = 1 − y
print("∫_0^1 f_Y(y) dy =", sp.integrate(f_U, (u_, 0, 1)))
E_Y = sp.integrate((1 - u_) * f_U, (u_, 0, 1))                  # E[Y] = ∫ y f_Y(y) dy with y = 1 − u
Var_Y = sp.simplify(sp.integrate((1 - u_) ** 2 * f_U, (u_, 0, 1)) - E_Y ** 2)
E_of_E = sp.integrate(E_cond, (xs_, 0, 1))                     # E[E[Y|X]]: average (1 + x)/2 over X ~ U(0, 1)
within = sp.integrate(V_cond, (xs_, 0, 1))                      # E[Var(Y|X)]
between = sp.integrate(E_cond ** 2, (xs_, 0, 1)) - E_of_E ** 2  # Var(E[Y|X])
print(f"E[Y] from f_Y = {E_Y};  E[E[Y|X]] = {E_of_E}")
print(f"Var(Y) from f_Y = {Var_Y};  E[Var(Y|X)] + Var(E[Y|X]) = {within} + {between} = {within + between} ≈ {float(within + between):.4f}")
print(f"SIMULATED: mean of Y = {Y_sim.mean():.4f}, variance of Y = {Y_sim.var():.4f}")

🔍 **What just happened?** The marginal density $-\ln(1 - y)$ integrates to 1, and from it SymPy finds $E[Y] = 3/4$ and
$\operatorname{Var}(Y) = 7/144$. The two laws give the same answers without ever computing $f_Y$: the average of the
conditional means is $3/4$, and the within-group part $1/36$ plus the between-group part $1/48$ is $7/144 \approx 0.0486$.
The 200,000 simulated values agree (mean 0.750, variance 0.049).

**Why biologists care: mixtures of cell types.** Let $T$ be the cell type of a randomly chosen cell and $X$ the count of
a marker gene. Within one type, counts are Poisson with mean $\lambda_T$, so $E[X \mid T] = \operatorname{Var}(X \mid T) =
\lambda_T$ ([Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb)). The laws give $E[X] = \sum_t P(t)\lambda_t$ (Section 3) and

$$\operatorname{Var}(X) = \underbrace{E[\lambda_T]}_{\text{within types}} + \underbrace{\operatorname{Var}(\lambda_T)}_{\text{between types}} = E[X] + \operatorname{Var}(\lambda_T).$$

*In words:* in a mixed population the variance of a gene equals its mean (Poisson noise) **plus** the variance of its
true level across cell types. A gene with the same level in every type has variance ≈ mean; a marker gene, high in one
type and low in the others, has a large between-type term — the mathematical basis of highly-variable-gene selection.

In [ ]:
# ── 9.3 The law of total variance in SIMULATED single-cell counts: within + between ──
for gene in ["CD3E", "COL1A1", "bg05"]:
    lam = sc["rates"][gene]                                       # true λ of the gene in each cell type
    p_t = share[lam.index]                                        # P(type)
    within_t = np.sum(p_t * lam)                                  # E[Var(X | T)] = E[λ_T] (Poisson)
    between_t = np.sum(p_t * lam ** 2) - np.sum(p_t * lam) ** 2   # Var(E[X | T]) = Var(λ_T)
    print(f"{gene:7s}: E[X] = {within_t:6.3f} | Var(X): within {within_t:6.3f} + between {between_t:6.3f} = "
          f"{within_t + between_t:7.3f} | sample variance across 440 cells {counts[gene].var():7.3f}")

🔍 **What just happened?** For the T-cell marker CD3E and the fibroblast marker COL1A1, the between-type term is several
times larger than the within-type (Poisson) term, so their total variance is far above their mean. For the background
gene bg05, whose true level is similar in all four cell types, the between-type term is tiny and the variance is close
to the mean. The sample variances across the 440 simulated cells match the law-of-total-variance predictions to within
sampling noise. Section 10 turns this into a gene-selection rule.

✏️ **Your turn.** (1) For the three-toss example, verify $E[Y] = E[E[Y \mid X]] = \tfrac12\cdot1 + \tfrac12\cdot2 = 1.5$
and $\operatorname{Var}(Y) = E[\operatorname{Var}(Y \mid X)] + \operatorname{Var}(E[Y \mid X]) = \tfrac12 + \tfrac14 =
\tfrac34$ (compare with Binomial(3, ½)). (2) In the continuous example, compute $E[Y \mid X = 0.5]$ and
$\operatorname{Var}[Y \mid X = 0.5]$ and check them in the binned simulation.

### 🌉 Bridge — mixtures, latent variables and the EM algorithm

The law of total variance is the heart of **mixture models**: a population made of hidden groups (cell types, tumour
subtypes, disease states) has a distribution whose mean is the average of the group means and whose variance is
"within + between". The **EM algorithm** of 02-620 ([BiologyToML 13 · Mixture models and EM](../../../../ml/tutorial/BiologyToML/notebooks/13_mixture_models_em.ipynb)) estimates the hidden group of every data point using
exactly the conditional expectations of this section, and Gaussian mixture clustering of cells is the continuous version
of the cell-type example above. Conditional expectation $E[Y \mid X = x]$ is also what every regression model estimates
— a straight line in linear regression ([Module 27 · 📉 Linear regression: the whole course in one model](27_linear_regression_review.ipynb)).

### 📘 Lecture notes — conditional expectation and variance

**Discrete.** $E[X \mid Y = y] = \sum_x x\,p(X = x \mid Y = y)$;
$\operatorname{Var}[X \mid Y = y] = \sum_x (x - E[X \mid Y = y])^2\,p(X = x \mid Y = y)$.
**Continuous.** $E[X \mid Y = y] = \int x\,f(x \mid y)\,dx$; $\operatorname{Var}[X \mid Y = y] = \int (x - E[X \mid Y =
y])^2 f(x \mid y)\,dx$.

**Worked example.** $X \sim \text{Uniform}(0, 1)$, $Y \mid X = x \sim \text{Uniform}(x, 1)$, $f(y \mid x) = \frac{1}{1 - x}$:

$$E[Y \mid X = x] = \frac{1}{1 - x}\Big[\tfrac12y^2\Big]_x^1 = \frac{1 - x^2}{2(1 - x)} = \frac{1 + x}{2}, \qquad \operatorname{Var}[Y \mid X = x] = \frac{1 + x + x^2}{3} - \frac{(1 + x)^2}{4} = \frac{(1 - x)^2}{12}.$$

*In words:* the midpoint and the uniform variance of the interval $(x, 1)$; the variance derivation expands
$[y - \tfrac{1 + x}{2}]^2$, integrates term by term, evaluates at $y = 1$ and $y = x$, and cancels the factor $1 - x$.

**Laws.** $E[Y] = E[E[Y \mid X]]$ (here $3/4$); $\operatorname{Var}(Y) = E[\operatorname{Var}(Y \mid X)] +
\operatorname{Var}(E[Y \mid X])$ (here $1/36 + 1/48 = 7/144$). For Poisson counts in a mixture of cell types:
$\operatorname{Var}(X) = E[X] + \operatorname{Var}(\lambda_T)$.

> 🎓 **Exam tip.** In $\int \dots dy$ the conditioning value $x$ is a constant — including in the limits of integration.
> Check a conditional-variance answer by plugging in an easy value ($x = 0$ gives Uniform(0, 1), variance $1/12$).

---
## 10 · Variance reveals differential expression: highly variable genes

> 📍 **Where we are.** Everything comes together in one of the first steps of every single-cell analysis.

Let $X$ be the expression of a gene $G$ across different cell types or conditions. If $\operatorname{Var}(X)$ is
**high**, the expression varies — the gene is a possible **marker gene** (or responds to the condition); if
$\operatorname{Var}(X) \approx 0$, the gene is uniformly expressed — a **housekeeping gene**. In biology this is used
for **feature selection in single-cell RNA-seq** and for prioritising genes for clustering and dimensionality reduction
(e.g. **Seurat's highly-variable-gene selection**; Satija *et al.*, 2015). We look at both of our data sets.

In [ ]:
# ── 10.1 Variance of every gene across the 24 SIMULATED infection samples ──
gene_var = expr_df.var(ddof=1).sort_values(ascending=False)
infection_genes = [g for genes in INFECTION_PATHWAYS.values() for g in genes]
rank = pd.Series(np.arange(1, len(gene_var) + 1), index=gene_var.index)          # 1 = most variable
print("10 most variable genes:", gene_var.index[:10].tolist())
print("ranks of the 12 infection-pathway genes:", sorted(rank[infection_genes].tolist()))
print("ranks of the 12 housekeeping genes:     ", sorted(rank[HOUSEKEEPING_GENES].tolist()),
      "| their variances:", gene_var[HOUSEKEEPING_GENES].round(4).min(), "to", gene_var[HOUSEKEEPING_GENES].round(4).max())
control = expr["samples"].condition.to_numpy() == "control"
var_control = expr_df[control].var(ddof=1)
for g in ["g160", "IL6", "STAT1", "MX1"]:
    print(f"  {g:5s}: variance across all 24 samples {gene_var[g]:.2f} | across the 12 control samples only {var_control[g]:.2f}")

fig, ax = plt.subplots(figsize=(7, 3.4))
ranks = np.arange(1, len(gene_var) + 1)
ax.semilogy(ranks, gene_var.to_numpy(), color="0.6", lw=1.5, label="all 400 genes")
is_inf = gene_var.index.isin(infection_genes)
is_hk = gene_var.index.isin(HOUSEKEEPING_GENES)
ax.semilogy(ranks[is_inf], gene_var.to_numpy()[is_inf], "o", color="tab:red", ms=5, label="infection-pathway genes")
ax.semilogy(ranks[is_hk], gene_var.to_numpy()[is_hk], "s", color="tab:blue", ms=5, label="housekeeping genes")
ax.set_xlabel("rank of the gene by variance"); ax.set_ylabel("variance across samples (log2², log scale)")
ax.set_title("variance separates responding genes from housekeeping genes"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Ranking the 400 genes by their variance across the 24 samples puts all 12 infection-pathway
genes among the 65 most variable (IL6 is second) and the 12 **housekeeping genes at the very bottom** (ranks 389–400),
with variances of only 0.002–0.010 — measurement noise. A variance ranking alone, with no labels, separates "genes that
change between samples" from "genes that do not". It does **not** say *why* a gene changes: the most variable gene, g160,
is almost as variable among the 12 control samples alone (variance 4.94 versus 5.21), because its baseline level differs
between tissues, whereas the variance of STAT1 (1.13 → 0.15) and MX1 (1.70 → 0.23) collapses once the infected samples
are removed — their variance is a between-condition term
in the sense of the law of total variance (Section 9).

### 10.2 The mean–variance relationship of counts

For counts, variance must be judged **relative to the mean**: a Poisson count has variance equal to its mean ([Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb)),
so a highly expressed gene is automatically more variable. Real single-cell data are **over-dispersed**: with a
cell-to-cell factor in the rate, the variance is $\mu + \phi\mu^2$ (the course simulator's `overdispersion` $\phi$).
Seurat-style selection therefore ranks genes by a **dispersion** such as $\operatorname{Var}/\text{mean}$ rather than by
the raw variance.

In [ ]:
# ── 10.2 Mean–variance plots: one cell type, all cells, over-dispersed counts; dispersion ranking (SIMULATED) ──
markers = [g for genes in CELL_TYPE_MARKERS.values() for g in genes]
counts_od = pd.DataFrame(sc_od["counts"], columns=sc_od["genes"])
t_cells = counts[sc["labels"] == "T cell"]
panels = [("Poisson counts, T cells only", t_cells), ("Poisson counts, all 440 cells", counts),
          ("over-dispersed counts (φ = 0.3), T cells only", counts_od[sc_od["labels"] == "T cell"])]
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), sharex=True, sharey=True)
mu_grid = np.logspace(-1.3, 1.4, 50)
for ax, (title, data) in zip(axes, panels):
    m, v = data.mean(), data.var()
    is_marker = m.index.isin(markers)
    ax.loglog(m[~is_marker], v[~is_marker], "o", color="0.5", ms=4, label="background genes")
    ax.loglog(m[is_marker], v[is_marker], "o", color="tab:red", ms=4, label="marker genes")
    ax.loglog(mu_grid, mu_grid, "k--", lw=1, label="Var = mean (Poisson)")
    if "over" in title:
        ax.loglog(mu_grid, mu_grid + 0.3 * mu_grid ** 2, color="tab:blue", lw=1, label="Var = μ + 0.3μ²")
    ax.set_title(title, fontsize=9); ax.set_xlabel("mean count")
axes[0].set_ylabel("variance of the count"); axes[2].legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

dispersion = (counts.var() / counts.mean()).sort_values(ascending=False)
top12 = dispersion.index[:12].tolist()
print("top 12 genes by dispersion Var/mean across all cells:", top12)
print(f"marker genes among them: {sum(g in markers for g in top12)} of 12;  "
      f"median dispersion of background genes {dispersion[~dispersion.index.isin(markers)].median():.2f}")

🔍 **What just happened?** **Left:** within one cell type, Poisson counts lie on the dashed line Var = mean — marker genes
included (red), whose high or low level is constant within T cells. **Middle:** across all 440 cells, the background
genes stay on the line but the marker genes jump far above it — their variance contains the between-type term
$\operatorname{Var}(\lambda_T)$ of Section 9. **Right:** with over-dispersion even a single cell type lies above the
Poisson line, along the curve $\mu + 0.3\mu^2$ (blue) — this extra variance is "biological noise", not cell-type
structure, which is why real pipelines model it before ranking genes. Ranking genes by dispersion (variance / mean)
across all cells returns all 12 marker genes as the top 12, while background genes have a dispersion near 1.

✏️ **Your turn.** (1) Repeat the dispersion ranking with the over-dispersed counts `counts_od`. Are the markers still on
top? (2) Use the law of total variance to split the variance of IL6 across the 24 samples into a part between the six
tissue × condition groups and a part within them. Which part dominates?

### 🌉 Bridge — highly variable genes and count models in single-cell analysis

Highly-variable-gene selection is the first step of standard single-cell workflows (Seurat, Scanpy): typically the
2,000 genes with the largest **standardised dispersion** are kept for PCA, clustering and visualisation ([BiologyToML 30 · Single-cell RNA-seq](../../../../ml/tutorial/BiologyToML/notebooks/30_single_cell_rnaseq.ipynb)). The
mean–variance relationship of Section 10.2 is modelled explicitly by **negative-binomial** count models, whose variance
$\mu + \phi\mu^2$ is exactly the over-dispersed curve above; they are the basis of differential-expression tools and of
generalised linear models for counts ([BiologyToML 24 · GLMs for count data](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb)). The moments of this module — means, variances, covariances — are also
the language of [BiologyToML 02 · Probability for biologists](../../../../ml/tutorial/BiologyToML/notebooks/02_probability_for_biologists.ipynb).

## 📌 Key takeaways

- **Expectation** $E[X] = \sum x\,p(x)$ or $\int x f(x)\,dx$: the long-run average and the centre of gravity (two
  4-sided dice → 5; one die → 3.5, not an outcome; Bernoulli → $p$; Uniform(−1, 3) → 1). It may fail to exist for
  heavy-tailed unbounded variables.
- $E[h(X)] = \sum h(x)p(x)$ or $\int h(x)f(x)\,dx$, but $E[h(X)] \ne h(E[X])$ in general ($E[X^2] = \tfrac13$ vs
  $\tfrac14$; mean of logs ≠ log of mean).
- From data: the **empirical distribution**'s expectation is the **sample mean**; average expression estimates $E[X]$.
- **Linearity:** $E[aX + b] = aE[X] + b$ and $E[\sum X_i] = \sum E[X_i]$ — **no independence needed**; indicators turn
  counting into adding probabilities.
- **Variance** $E[(X - \mu)^2] = E[X^2] - \mu^2$, standard deviation in the units of $X$; one rare value can dominate it
  (buggy simulator: 9.88 of 10.14). $\operatorname{Var} \ge 0$, $= 0$ only for constants.
- $\operatorname{Var}(aX + b) = a^2\operatorname{Var}(X)$; $\operatorname{Var}(X + Y) = \operatorname{Var}X +
  \operatorname{Var}Y + 2\operatorname{Cov}$; variances add only for independent (uncorrelated) variables;
  $\operatorname{Var}(\bar X) = \sigma^2/n$. The sample variance divides by $n - 1$.
- **Skewness** and **kurtosis** describe shape; raw expression is right-skewed and the log transform symmetrises it.
- **Conditional** moments: $E[Y \mid X = x] = (1 + x)/2$, $\operatorname{Var}[Y \mid X = x] = (1 - x)^2/12$ for
  $Y \mid X \sim U(X, 1)$; $E[Y] = E[E[Y \mid X]]$ and $\operatorname{Var}Y = E[\operatorname{Var}(Y \mid X)] +
  \operatorname{Var}(E[Y \mid X])$.
- **Highly variable genes** = large between-group variance; housekeeping genes have variance ≈ 0; counts must be judged
  relative to the Poisson line Var = mean.

---
## ✅ Checkpoint

1. A random variable takes the values 1, 2, 4 with probabilities 0.5, 0.3, 0.2. Compute $E[X]$, $E[X^2]$ and $\operatorname{Var}(X)$.
   <details><summary>Answer</summary>E[X] = 0.5 + 0.6 + 0.8 = 1.9. E[X²] = 0.5 + 1.2 + 3.2 = 4.9. Var = 4.9 − 3.61 = 1.29.</details>
2. Compute $E[X]$ and $\operatorname{Var}(X)$ for $X \sim \text{Uniform}(2, 6)$ by integration.
   <details><summary>Answer</summary>f = 1/4 on (2, 6). E[X] = ∫₂⁶ x/4 dx = [x²/8]₂⁶ = 36/8 − 4/8 = 4. E[X²] = ∫₂⁶ x²/4 dx = [x³/12]₂⁶ = 216/12 − 8/12 = 52/3. Var = 52/3 − 16 = 4/3 = (6 − 2)²/12.</details>
3. $E(X) = 5$ and $\operatorname{Var}(X) = 2$. Compute $E(3X - 5)$, $E(-3X + 15)$, $\operatorname{Var}(3X - 5)$ and $\operatorname{Var}(-3X + 15)$.
   <details><summary>Answer</summary>E(3X − 5) = 10; E(−3X + 15) = 0; Var(3X − 5) = 9·2 = 18; Var(−3X + 15) = (−3)²·2 = 18 — the sign of a and the shift b do not matter for the variance.</details>
4. Prove $\operatorname{Var}(X) = E(X^2) - [E(X)]^2$.
   <details><summary>Answer</summary>With μ = E(X): Var = E[(X − μ)²] = E(X² − 2μX + μ²) = E(X²) − 2μE(X) + μ² = E(X²) − 2μ² + μ² = E(X²) − μ², using linearity of expectation.</details>
5. $X$ is the face of a fair die and $Y = 7 - X$. Compute $E[X + Y]$ and $\operatorname{Var}(X + Y)$, and explain why linearity of expectation still holds although $X$ and $Y$ are dependent.
   <details><summary>Answer</summary>X + Y = 7 always: E = 7 = 3.5 + 3.5 and Var = 0. Linearity of expectation never uses independence (the proof only splits an integral or a sum). The variance rule fails: Var X + Var Y = 35/6, but Cov(X, Y) = −35/12 contributes 2·(−35/12) = −35/6.</details>
6. Recompute $E[S]$ and $\operatorname{Var}[S]$ for the buggy simulator with the shortcut formula, and say how much of the variance comes from the value 100.
   <details><summary>Answer</summary>E[S] = 0.4995 + 0.1 = 0.5995. E[S²] = 0.4995·1 + 0.001·10,000 = 10.4995. Var = 10.4995 − 0.5995² = 10.4995 − 0.3594 = 10.1401. The 100 term contributes 0.001·(100 − 0.5995)² = 9.880, about 97 %.</details>
7. Why does the sample variance divide by $n - 1$? What is $E\big[\frac1n\sum(X_i - \bar X)^2\big]$ for $n = 4$ and $\sigma^2 = 8$?
   <details><summary>Answer</summary>Deviations are measured from X̄, which is fitted to the same data, so they are too small on average: E[Σ(Xᵢ − X̄)²] = (n − 1)σ². For n = 4, σ² = 8: E = (3/4)·8 = 6, whereas the n − 1 version averages 8.</details>
8. You average 9 independent technical replicates, each with standard deviation 0.6 log2 units. What is the standard deviation of the average?
   <details><summary>Answer</summary>Var(X̄) = σ²/n = 0.36/9 = 0.04, so SD = 0.2 = 0.6/√9.</details>
9. For $X \sim \text{Uniform}(0, 1)$ and $Y \mid X = x \sim \text{Uniform}(x, 1)$, compute $E[Y \mid X = 0.2]$ and $\operatorname{Var}[Y \mid X = 0.2]$, and use the law of total expectation to find $E[Y]$.
   <details><summary>Answer</summary>E[Y | X = 0.2] = 1.2/2 = 0.6; Var = 0.8²/12 = 0.0533. E[Y] = E[(1 + X)/2] = (1 + 0.5)/2 = 0.75.</details>
10. A gene is expressed at Poisson rate 2 in half of the cells (type A) and rate 10 in the other half (type B). Compute $E[X]$ and $\operatorname{Var}(X)$ for a random cell. Is the gene a good candidate marker?
    <details><summary>Answer</summary>E[X] = 0.5·2 + 0.5·10 = 6. Within: E[λ] = 6; between: Var(λ) = 0.5·4 + 0.5·100 − 36 = 16. Var(X) = 6 + 16 = 22 ≫ mean 6 (dispersion 3.7) — yes, its variance comes mostly from the difference between the types.</details>
11. What is the skewness of a symmetric distribution, and why is raw expression right-skewed?
    <details><summary>Answer</summary>0, because positive and negative cubed deviations cancel. Most genes are expressed at low levels and a few at very high levels on the linear scale (expression is roughly log-normal), giving a long right tail; the log transform makes it roughly symmetric.</details>
12. In NumPy, `np.var(x)` and in pandas `pd.Series(x).var()` give different numbers. Why?
    <details><summary>Answer</summary>NumPy divides by n (ddof = 0) by default, pandas by n − 1 (ddof = 1). Use `np.var(x, ddof=1)` for the unbiased sample variance.</details>

## 🧠 Questions & discussion

1. The expected number of offspring of an organism is 1.05. Does that mean a lineage is guaranteed to survive? What
   other moment (or which whole distribution) would you need to judge its risk of extinction?
2. Mean expression is often reported on the log scale (mean of log2 values) and fold changes as differences of those
   means. What is the biological meaning of the mean of logs (the geometric mean)? When could it mislead?
3. One sample in an experiment of 12 has a value 50 times larger than the others. Which summaries (mean, median,
   variance, rank-based statistics) are affected, and what should you do before deciding to remove it?
4. Highly variable genes are selected across all cells. Could a gene be highly variable for technical reasons (batch,
   sequencing depth) rather than biological ones? How would the law of total variance help you separate the sources?
5. Why is it acceptable to add expectations of dependent variables but not their variances? Give a biological example
   where ignoring a positive covariance would make you over-confident.

## 🚀 Bonus — going further

- **Chebyshev's inequality.** For any $X$ with mean $\mu$ and variance $\sigma^2$, $P(|X - \mu| \ge k\sigma) \le 1/k^2$.
  Check it for the buggy simulator, a die and a normal variable. How conservative is it?
- **Moment generating functions.** $M_X(t) = E[e^{tX}]$ "generates" all moments: $E[X^k] = M_X^{(k)}(0)$. Use SymPy to find
  the MGF of a Poisson variable and differentiate it to recover the mean and variance.
- **Median versus mean.** Compute the median and the mean of the raw expression values of Section 8. Which is more
  robust to the long tail? Repeat after adding one outlier.
- **Further reading.** DeGroot & Schervish, *Probability and Statistics*, Chapter 4 (expectation, variance, moments,
  covariance, conditional expectation); Wasserman, *All of Statistics*, Chapter 3.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 21 — Expectation, variance and moments — date:
Input: dice, coins, the buggy coin simulator; uniform and conditional-uniform variables; SIMULATED infection
       expression (24 × 400) and single-cell counts (440 × 40, Poisson and over-dispersed)
Output: expectations, variances, standard deviations, sample-variance bias, skewness/kurtosis,
        E[Y|X], Var[Y|X], total expectation/variance, variance ranks of genes, mean–variance plots
Method: Σ x p(x), ∫ x f(x) dx, E[h(X)], linearity, Var = E[X²] − E[X]², Var(aX+b), Var of sums, laws of total E/Var
Key equation: E[aX + b] = aE[X] + b;  Var(X) = E[X²] − (E[X])²;  Var(Y) = E[Var(Y|X)] + Var(E[Y|X])
Key parameters: sample size n; ddof; overdispersion φ; the rare-value probability of the buggy simulator
What changed when I modified n / the probability of the value 100 / φ:
One failure mode: (e.g., Var(X+Y) without the covariance; np.var vs pandas .var; mean of logs vs log of means)
Connection to 02-680 (which part of the topic on expectation and moments this illustrates):
```

## 📚 References

- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — Chapter 4 (expectation, the
  centre-of-gravity interpretation, properties of expectations and variances, moments, covariance and correlation,
  conditional expectation).
- Wasserman L. (2004) *All of Statistics*. Springer — Chapter 3 (expectation, variance, sample mean and variance,
  conditional expectation, the laws of total expectation and variance).
- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press — §6.4
  (summary statistics: means, covariances, empirical means).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences* — the chapters on expected value and variance.
- Anders S., McCarthy D. J., Chen Y., Okoniewski M., Smyth G. K., Huber W., Robinson M. D. (2013) Count-based
  differential expression analysis of RNA sequencing data using R and Bioconductor. *Nature Protocols* 8:1765–1786.
- Satija R., Farrell J. A., Gennert D., Schier A. F., Regev A. (2015) Spatial reconstruction of single-cell gene
  expression data. *Nature Biotechnology* 33:495–502.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.

## ➡️ Next

[Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb) — this module estimated expectations by sample means and showed, in several simulations, that
the estimates get better as $n$ grows, with $\operatorname{Var}(\bar X) = \sigma^2/n$. The next module makes this precise:
the **law of large numbers** (why sample means converge), the **central limit theorem** (why their distribution becomes
normal, whatever the data look like) and **confidence intervals** (how to report an estimate together with its
uncertainty) — the bridge from probability to statistics.